# Reto III.3 — Voyager 2 y Júpiter: **SOLO Sol, Júpiter y sonda**

**Cuaderno integral para Colab.** Incluye código Python completo, comparación de dos modelos, cálculo de invariantes y errores, estudio de radio de empalme, validación de Berg (2023), figuras, animación Plotly 2D, dos visores HTML con cámara libre **sin WebGL**, informe LaTeX y exportación ZIP.

**Solo requiere:** `horizons_jupiter.csv` y `horizons_voyager-2.csv`. **No solicita** `horizons_saturno.csv` ni incluye su gravedad. El valor **9.58 UA** es únicamente un **radio objetivo geométrico** para probar alcance radial.

**Orden recomendado:** ejecutar todas las celdas, de arriba abajo. No se necesita GPU.

In [1]:
# 1. Instalar dependencias e iniciar en /content
%pip -q install numpy scipy pandas matplotlib plotly
import os
from pathlib import Path
os.chdir('/content')
print('Carpeta de trabajo:', Path.cwd())

Carpeta de trabajo: /content


In [2]:
# 2. Cargar únicamente las DOS efemérides solicitadas (si aún faltan)
from pathlib import Path
from google.colab import files
NECESARIOS = ['horizons_jupiter.csv', 'horizons_voyager-2.csv']
faltantes = [s for s in NECESARIOS if not Path('/content', s).is_file()]
if faltantes:
    print('Selecciona estos archivos:', faltantes)
    uploads = files.upload()
    for nombre, datos in uploads.items():
        Path('/content', nombre).write_bytes(datos)
else:
    print('Las dos efemérides ya están disponibles; no es necesario recargarlas.')
for nombre in NECESARIOS:
    assert Path('/content', nombre).is_file(), f'Falta {nombre}'
print('Archivos de entrada:', NECESARIOS)

Las dos efemérides ya están disponibles; no es necesario recargarlas.
Archivos de entrada: ['horizons_jupiter.csv', 'horizons_voyager-2.csv']


## 2. Motor físico y matemático

Este bloque constituye el **código científico completo**. Las funciones implementan las dos formulaciones: cónicas empalmadas (solar → jovicéntrica → solar) e integración heliocéntrica restringida Sol+Júpiter con corrección indirecta. Júpiter y Voyager se leen de Horizons; Saturno no interviene. No ejecutes una versión vieja del script encima de esta celda.

In [3]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""Reto III.3: asistencia gravitacional de Voyager 2 en Júpiter.

Modelo 1: cónicas empalmadas (Sol | Júpiter | Sol).
Modelo 2: sonda restringida en campos heliocéntricos de Sol + Júpiter;
          con Júpiter prescrito por efemérides Horizons. No se integra
          un sistema autónomo de tres cuerpos y no se incluye Saturno.

Unidades internas: km, s, km/s, km^3/s^2 y km^2/s^2.
Fechas: escala TDB de Horizons, representadas como fechas civiles aproximadas.
Requisitos: numpy, scipy, pandas, matplotlib; plotly es opcional.

Uso:
    python asistencia_gravitacional_refactorizada.py \
        --jupiter horizons_jupiter.csv \
        --voyager horizons_voyager-2.csv

Autor: refactorización científica para el Reto III.3, noviembre de 2026.
"""
from __future__ import annotations

import argparse
import json
from dataclasses import dataclass, field
from io import StringIO
from pathlib import Path
from typing import Any, Callable

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from numpy.typing import NDArray
from scipy.integrate import solve_ivp
from scipy.interpolate import CubicHermiteSpline
from scipy.optimize import brentq, minimize_scalar

Array = NDArray[np.float64]
DAY = 86400.0
AU = 149_597_870.7


@dataclass(frozen=True)
class Physics:
    """Constantes físicas coherentes en kilómetros y segundos."""
    mu_sun: float = 1.32712440018e11
    mu_jupiter: float = 1.26686534e8
    mass_sun: float = 1.98847e30
    mass_jupiter: float = 1.89813e27
    a_jupiter: float = 778.57e6
    target_radius_au: float = 9.58  # solo referencia geométrica, no planeta físico

    @property
    def r_soi(self) -> float:
        """Radio de influencia de Laplace (masa jovicéntrica/masa solar)."""
        return self.a_jupiter * (self.mass_jupiter / self.mass_sun) ** (2 / 5)

    @property
    def r_hill(self) -> float:
        """Radio de Hill aproximado suponiendo órbitas circulares."""
        return self.a_jupiter * (self.mass_jupiter / (3 * self.mass_sun)) ** (1 / 3)


@dataclass(frozen=True)
class Settings:
    """Opciones de integración y exportación."""
    rtol: float = 3e-10
    atol_pos: float = 1e-3
    atol_vel: float = 2e-11
    max_step_days: float = 0.75
    days_before_entry: float = 100.0
    days_after_target_cross: float = 180.0
    sensitivity_factors: tuple[float, ...] = (0.50, 0.65, 0.80, 1.00)
    draw_dpi: int = 170

    @property
    def atol(self) -> Array:
        return np.array([self.atol_pos] * 3 + [self.atol_vel] * 3)


@dataclass(frozen=True)
class State:
    """Estado tridimensional de posición y velocidad."""
    r: Array
    v: Array

    def pack(self) -> Array:
        return np.r_[self.r, self.v].astype(float)

    @staticmethod
    def unpack(y: Array) -> State:
        return State(np.asarray(y[:3]), np.asarray(y[3:6]))


@dataclass(frozen=True)
class Ephemeris:
    """Interpolador Hermite de posición y velocidad de Horizons.

    A diferencia de dos splines independientes, v(t)=dr(t)/dt.
    """
    name: str
    t: Array
    jd: Array
    dates: pd.DatetimeIndex
    r_data: Array
    v_data: Array
    spline: Any = field(repr=False)

    def state(self, t: float) -> State:
        if not (self.t[0] <= t <= self.t[-1]):
            raise ValueError(f"Tiempo fuera de efemérides para {self.name}: {t / DAY:.2f} días")
        return State(np.asarray(self.spline(t)), np.asarray(self.spline(t, 1)))

    def positions(self, t: Array) -> Array:
        return np.asarray(self.spline(t))

    def velocities(self, t: Array) -> Array:
        return np.asarray(self.spline(t, 1))


@dataclass(frozen=True)
class Encounter:
    """Cruces de SOI y periapsis calculados sobre efemérides reales."""
    t_in: float
    t_peri: float
    t_out: float
    r_peri: float
    radius: float


@dataclass
class Orbit:
    """Propagación definida a trozos, evaluable en tiempo absoluto."""
    name: str
    pieces: list[tuple[float, float, Any]]
    t_in: float
    t_out: float | None = None

    def state(self, t: float) -> State:
        for t0, t1, sol in self.pieces:
            if min(t0, t1) - 1e-5 <= t <= max(t0, t1) + 1e-5:
                return State.unpack(np.asarray(sol.sol(t)))
        raise ValueError(f"t={t/DAY:.4f} no pertenece a la órbita {self.name}")

    def states(self, times: Array) -> tuple[Array, Array]:
        y = np.array([self.state(float(tt)).pack() for tt in times])
        return y[:, :3], y[:, 3:]


@dataclass(frozen=True)
class OrbitalElements:
    """Elementos osculatrices heliocéntricos de dos cuerpos."""
    a_au: float
    eccentricity: float
    aphelion_au: float
    inclination_deg: float
    energy: float
    h: Array


def horizons_csv(path: Path, name: str, jd_ref: float | None = None) -> Ephemeris:
    """Lee Horizons en formato bruto ($$SOE/$$EOE) o CSV tabular limpio.

    Comprueba que el centro es el Sol, el sistema eclíptico J2000 y KM-S,
    cuando esas cabeceras existen. No combina marcos distintos inadvertidamente.
    """
    if not path.exists():
        raise FileNotFoundError(path)
    raw = path.read_text(encoding="utf-8-sig")
    if "$$SOE" in raw and "$$EOE" in raw:
        hdr, rem = raw.split("$$SOE", 1)
        payload = rem.split("$$EOE", 1)[0]
        for phrase in ("Center body name: Sun (10)", "Output units    : KM-S",
                       "Reference frame : Ecliptic of J2000.0"):
            if phrase not in hdr:
                raise ValueError(f"{path.name}: encabezado Horizons incompatible: {phrase}")
        lines = [ln.strip().strip('"') for ln in payload.splitlines() if ln.strip()]
        df = pd.read_csv(StringIO("\n".join(lines)), header=None,
                         usecols=range(8), skipinitialspace=True,
                         names=["JDTDB", "Calendar Date (TDB)", "X", "Y", "Z", "VX", "VY", "VZ"])
    else:
        df = pd.read_csv(path)
        df.columns = df.columns.str.strip().str.strip('"')
    cols = ["JDTDB", "Calendar Date (TDB)", "X", "Y", "Z", "VX", "VY", "VZ"]
    if not set(cols).issubset(df.columns):
        raise ValueError(f"{path.name}: faltan columnas {set(cols) - set(df.columns)}")
    df = df[cols].copy()
    for c in ["JDTDB", "X", "Y", "Z", "VX", "VY", "VZ"]:
        df[c] = pd.to_numeric(df[c], errors="coerce")
    fechas = df["Calendar Date (TDB)"].astype(str).str.strip()
    fechas = fechas.str.replace(r"^A\.D\.\s*", "", regex=True)
    dates = pd.to_datetime(fechas, format="%Y-%b-%d %H:%M:%S.%f", errors="coerce")
    df["_date"] = dates
    df = df.dropna().sort_values("JDTDB").drop_duplicates("JDTDB")
    if len(df) < 20:
        raise ValueError(f"{path.name}: insuficientes registros")
    jd = df["JDTDB"].to_numpy(dtype=float)
    tref = jd[0] if jd_ref is None else jd_ref
    t = (jd - tref) * DAY
    r = df[["X", "Y", "Z"]].to_numpy(dtype=float)
    v = df[["VX", "VY", "VZ"]].to_numpy(dtype=float)
    spline = CubicHermiteSpline(t, r, v, axis=0)
    return Ephemeris(name, t, jd, pd.DatetimeIndex(df["_date"]), r, v, spline)


def synchronize_data(paths: dict[str, Path]) -> dict[str, Ephemeris]:
    """Carga las efemérides sobre la misma época Juliana."""
    first = horizons_csv(paths["jupiter"], "Júpiter")
    result = {"jupiter": first}
    result["voyager"] = horizons_csv(paths["voyager"], "Voyager 2", first.jd[0])
    for name, eph in result.items():
        if not np.array_equal(first.jd, eph.jd):
            raise ValueError(f"Mallas JDTDB distintas para {name}; genere los CSV con iguales fechas")
    return result


def unit(v: Array) -> Array:
    """Devuelve el vector unitario, rechazando la norma nula."""
    n = np.linalg.norm(v)
    if n == 0:
        raise ValueError("Vector de norma cero")
    return v / n


def angle_deg(a: Array, b: Array) -> float:
    """Ángulo tridimensional entre vectores, en grados."""
    return float(np.degrees(np.arccos(np.clip(np.dot(unit(a), unit(b)), -1., 1.))))


def orbit_elements(s: State, mu: float) -> OrbitalElements:
    """Calcula elementos osculatrices; afelio infinito para órbita abierta."""
    rnorm = np.linalg.norm(s.r)
    h = np.cross(s.r, s.v)
    energy = float(np.dot(s.v, s.v) / 2 - mu / rnorm)
    evec = np.cross(s.v, h) / mu - s.r / rnorm
    ecc = float(np.linalg.norm(evec))
    semimajor = float(-mu / (2 * energy)) if abs(energy) > 1e-14 else np.inf
    aphelion = semimajor * (1 + ecc) if energy < 0 else np.inf
    inc = float(np.degrees(np.arccos(np.clip(h[2] / np.linalg.norm(h), -1., 1.))))
    return OrbitalElements(semimajor / AU, ecc, aphelion / AU, inc, energy, h)


def find_crossing(eph: Ephemeris, level: float, start: float, end: float,
                  direction: int = 1) -> float | None:
    """Primer cruce de distancia heliocéntrica de un nivel."""
    knots = eph.t[(eph.t >= start) & (eph.t <= end)]
    ts = np.unique(np.r_[start, knots, end])
    d = np.linalg.norm(eph.positions(ts), axis=1) - level
    inds = np.flatnonzero((d[:-1] <= 0) & (d[1:] >= 0)) if direction == 1 else np.flatnonzero((d[:-1] >= 0) & (d[1:] <= 0))
    if not len(inds):
        return None
    k = int(inds[0])
    return float(brentq(lambda x: np.linalg.norm(eph.state(x).r) - level, ts[k], ts[k+1]))


def encounter_real(eph: dict[str, Ephemeris], radius: float) -> Encounter:
    """Determina entrada, periapsis y salida real respecto de Júpiter."""
    j, v = eph["jupiter"], eph["voyager"]
    d = np.linalg.norm(v.r_data - j.r_data, axis=1)
    ix_in = np.flatnonzero((d[:-1] > radius) & (d[1:] <= radius))
    if not len(ix_in):
        raise RuntimeError(f"No hay entrada a {radius:.3e} km")
    ix_out = np.flatnonzero((d[:-1] < radius) & (d[1:] >= radius))
    ix_out = ix_out[ix_out > ix_in[0]]
    if not len(ix_out):
        raise RuntimeError(f"No hay salida de {radius:.3e} km")
    def distance(t: float) -> float:
        return float(np.linalg.norm(v.state(t).r - j.state(t).r))
    k, l = int(ix_in[0]), int(ix_out[0])
    ti = brentq(lambda t: distance(t)-radius, j.t[k], j.t[k+1])
    to = brentq(lambda t: distance(t)-radius, j.t[l], j.t[l+1])
    res = minimize_scalar(distance, method="bounded", bounds=(ti, to), options={"xatol":0.01})
    return Encounter(float(ti), float(res.x), float(to), float(res.fun), radius)


def integration(rhs: Callable, t0: float, t1: float, y0: Array,
                settings: Settings, event: Callable | None = None) -> Any:
    """Integra con DOP853 y tolerancias vectoriales por variable."""
    kwargs = dict(method="DOP853", rtol=settings.rtol, atol=settings.atol,
                  max_step=settings.max_step_days * DAY, dense_output=True)
    if event is not None:
        kwargs["events"] = event
    sol = solve_ivp(rhs, (t0, t1), y0, **kwargs)
    if not sol.success:
        raise RuntimeError(f"Integración fallida: {sol.message}")
    return sol


def solar_rhs(phys: Physics) -> Callable:
    """Ecuaciones de Kepler alrededor del Sol (sin planetas)."""
    def rhs(t: float, y: Array) -> Array:
        r = y[:3]
        return np.r_[y[3:], -phys.mu_sun * r / np.linalg.norm(r)**3]
    return rhs


def jovian_rhs(phys: Physics) -> Callable:
    """Movimiento relativo de dos cuerpos respecto de Júpiter fijo."""
    def rhs(t: float, y: Array) -> Array:
        r = y[:3]
        return np.r_[y[3:], -phys.mu_jupiter * r / np.linalg.norm(r)**3]
    return rhs


def planet_rhs(phys: Physics, eph: dict[str, Ephemeris]) -> Callable:
    """Sonda sin masa con el Sol y planetas prescritos por Horizons."""
    def rhs(t: float, y: Array) -> Array:
        r = y[:3]
        a = -phys.mu_sun * r / np.linalg.norm(r)**3
        rj = eph["jupiter"].state(t).r
        dr = r-rj
        a -= phys.mu_jupiter * (dr / np.linalg.norm(dr)**3
                                + rj / np.linalg.norm(rj)**3)
        # El término -muJ*rJ/|rJ|³ es la corrección indirecta
        # por aceleración del origen heliocéntrico (Sol).
        # No se modela ni se consulta Saturno.
        return np.r_[y[3:], a]
    return rhs


def conics_orbit(eph: dict[str, Ephemeris], phys: Physics, cfg: Settings,
                 enc: Encounter, t_pre: float, t_end: float, name: str = "Modelo 1") -> Orbit:
    """Propagación Sol (atrás), Júpiter (SOI), Sol (hacia radio objetivo).

    Se ancla al estado real exactamente en la entrada de la SOI. La parte
    anterior se integra hacia atrás, sin forzarla a coincidir con Horizons.
    """
    s_entry = eph["voyager"].state(enc.t_in)
    j_entry = eph["jupiter"].state(enc.t_in)
    pre = integration(solar_rhs(phys), enc.t_in, t_pre, s_entry.pack(), cfg)
    rel0 = State(s_entry.r-j_entry.r, s_entry.v-j_entry.v)

    def exit_soi(t: float, y: Array) -> float:
        return float(np.linalg.norm(y[:3])-enc.radius)
    exit_soi.direction = 1
    exit_soi.terminal = True

    jov = integration(jovian_rhs(phys), enc.t_in, min(enc.t_in+300*DAY, t_end),
                      rel0.pack(), cfg, exit_soi)
    if len(jov.t_events[0]) != 1:
        raise RuntimeError("Modelo 1 no sale de la SOI")
    t_out = float(jov.t_events[0][0])
    relout = State.unpack(jov.sol(t_out))
    planet = eph["jupiter"].state(t_out)
    out = State(planet.r+relout.r, planet.v+relout.v)
    post = integration(solar_rhs(phys), t_out, t_end, out.pack(), cfg)

    # El tramo jovicéntrico se expone como solución heliocéntrica envolvente.
    class GalileanBridge:
        def sol(self, t: float) -> Array:
            rel = State.unpack(np.asarray(jov.sol(t)))
            planet_t = eph["jupiter"].state(float(t))
            return State(planet_t.r+rel.r, planet_t.v+rel.v).pack()
    return Orbit(name, [(t_pre, enc.t_in, pre), (enc.t_in, t_out, GalileanBridge()),
                        (t_out, t_end, post)], enc.t_in, t_out)


def forced_orbit(eph: dict[str, Ephemeris], phys: Physics, cfg: Settings,
                 enc: Encounter, t_pre: float, t_end: float) -> Orbit:
    """Modelo restringido Sol+Júpiter+sonda; Júpiter sigue Horizons."""
    rhs = planet_rhs(phys, eph)
    initial = eph["voyager"].state(enc.t_in).pack()
    backwards = integration(rhs, enc.t_in, t_pre, initial, cfg)
    forward = integration(rhs, enc.t_in, t_end, initial, cfg)
    name = "Modelo 2: Sol+Júpiter"
    return Orbit(name, [(t_pre, enc.t_in, backwards), (enc.t_in, t_end, forward)], enc.t_in)


def first_radius_time(orb: Orbit, start: float, end: float, radius: float) -> float | None:
    """Localiza el primer cruce ascendente de un radio orbital."""
    grid = np.linspace(start, end, int((end-start)/DAY)+2)
    d = np.array([np.linalg.norm(orb.state(float(t)).r)-radius for t in grid])
    hit = np.flatnonzero((d[:-1] <= 0) & (d[1:] >= 0))
    if not len(hit):
        return None
    k = int(hit[0])
    return float(brentq(lambda t: np.linalg.norm(orb.state(t).r)-radius, grid[k], grid[k+1]))


def two_body_invariants(positions: Array, velocities: Array, mu: float) -> tuple[Array, Array]:
    """Energía específica y momento angular vectorial (para un cuerpo central)."""
    en = np.einsum("ij,ij->i", velocities, velocities)/2 - mu/np.linalg.norm(positions, axis=1)
    return en, np.cross(positions, velocities)


def tidal_ratio(r: Array, rj: Array, phys: Physics) -> Array:
    """|a_S(rV)-a_S(rJ)|/|a_J(rV-rJ)|, sonda en marco de Júpiter."""
    rel = r-rj
    sun_s = -phys.mu_sun*r/np.linalg.norm(r, axis=1)[:,None]**3
    sun_j = -phys.mu_sun*rj/np.linalg.norm(rj, axis=1)[:,None]**3
    a_jov = phys.mu_jupiter/np.linalg.norm(rel,axis=1)**2
    return np.linalg.norm(sun_s-sun_j, axis=1)/a_jov


def jacobi_proxy(r: Array, v: Array, rj: Array, vj: Array, phys: Physics,
                 n: float, axis: Array) -> Array:
    """Constante de Jacobi dimensional aproximada en ejes baricéntricos.

    C=2U-|v_B|²+2 n (h_B·eje) con r_B=r−ηrJ, v_B=v−ηvJ,
    η=μJ/(μSol+μJ). Es exacta para órbitas circulares consistentes
    de los primarios y gravedad heliocéntrica con término indirecto.
    En Horizons (Júpiter no circular), C*(t) es solo un diagnóstico.
    """
    eta=phys.mu_jupiter/(phys.mu_sun+phys.mu_jupiter)
    rb=r-eta*rj
    vb=v-eta*vj
    rel=r-rj
    U = phys.mu_sun/np.linalg.norm(r,axis=1)+phys.mu_jupiter/np.linalg.norm(rel,axis=1)
    h=np.cross(rb,vb)
    return 2*U - np.einsum('ij,ij->i',vb,vb) + 2*n*(h@axis)


def hyperbola_geometry(relative: State, phys: Physics) -> dict[str, Any]:
    """Parámetros exactos de la hipérbola kepleriana osculatriz.

    b=|h|/v_inf, D=mu/(b v_inf²), gamma=2atan(D).
    Calcula vectores entrante y saliente en el infinito, no en la SOI.
    """
    r,v = relative.r, relative.v
    h_vec = np.cross(r,v)
    h = float(np.linalg.norm(h_vec))
    energy = float(np.dot(v,v)/2-phys.mu_jupiter/np.linalg.norm(r))
    if energy <= 0:
        raise RuntimeError("La energía jovicéntrica de entrada no describe una hipérbola")
    vinf = float(np.sqrt(2*energy))
    b = h/vinf
    ecc = float(np.sqrt(1 + 2*energy*h*h/phys.mu_jupiter**2))
    evec = np.cross(v,h_vec)/phys.mu_jupiter-r/np.linalg.norm(r)
    pvec = unit(evec)
    qvec = np.cross(unit(h_vec), pvec)
    f_inf = np.arccos(-1/ecc)
    def vel_asym(f: float) -> Array:
        return phys.mu_jupiter/h*(-np.sin(f)*pvec+(ecc+np.cos(f))*qvec)
    vin, vout = vel_asym(-f_inf), vel_asym(+f_inf)
    D = phys.mu_jupiter/(b*vinf**2)
    gamma = float(np.degrees(2*np.arctan(D)))
    rp = h*h/(phys.mu_jupiter*(1+ecc))
    return dict(energia_j=energy, h=h, b=b, D=D, vinf=vinf,
                e=ecc, gamma_deg=gamma, rp=rp, vin=vin, vout=vout)


def berg_parameters(geom: dict[str, Any], planet: State) -> dict[str, float]:
    """Ecuaciones (26),(27),(32),(37),(39)-(42) de P. Berg (2023).

    Adapta las componentes a base radial (y) y tangencial (x) en plano
    joviano, asumiendo movimiento casi circular (vJ radial ~ 0).
    Conserva por separado el resultado vectorial 3D sin esa hipótesis.
    """
    e_rad = unit(planet.r)
    e_tan = unit(planet.v - np.dot(planet.v,e_rad)*e_rad)
    u_planet = float(np.dot(planet.v,e_tan))
    vin, vout = geom["vin"], geom["vout"]
    vin2 = np.array([np.dot(vin,e_tan), np.dot(vin,e_rad)])
    vout2 = np.array([np.dot(vout,e_tan), np.dot(vout,e_rad)])
    v_in_hel = vin2 + np.array([u_planet,0.])
    v_out_hel = vout2 + np.array([u_planet,0.])
    u_i = float(np.linalg.norm(vin2))
    alpha = float(np.degrees(np.arctan2(v_in_hel[0],v_in_hel[1])))
    beta = float(np.degrees(np.arccos(np.clip(-vin2[0]/u_i,-1.,1.))))
    gamma = np.deg2rad(geom["gamma_deg"])
    beta_r = np.deg2rad(beta)
    # Berg (32) usa un giro orientado hacia adelante. Es válido sólo
    # si la hipérbola proyectada tiene el sentido de desvío de su figura.
    rad_in = u_i*u_i + u_planet*u_planet - 2*u_planet*u_i*np.cos(beta_r)
    rad_out = u_i*u_i + u_planet*u_planet - 2*u_planet*u_i*np.cos(beta_r+gamma)
    delta32 = float(np.sqrt(max(rad_out,0))-np.sqrt(max(rad_in,0)))
    D = geom["D"]
    cos_bg = ((1-D*D)*np.cos(beta_r)-2*D*np.sin(beta_r))/(1+D*D)
    delta37 = float(np.sqrt(max(u_i*u_i+u_planet*u_planet-2*u_planet*u_i*cos_bg,0)) - np.sqrt(max(rad_in,0)))
    delta_2d_vectors = float(np.linalg.norm(v_out_hel)-np.linalg.norm(v_in_hel))
    delta_3d = float(np.linalg.norm(planet.v+vout)-np.linalg.norm(planet.v+vin))
    signed = float(vin2[0]*vout2[1]-vin2[1]*vout2[0])
    # Si el giro proyectado NO concuerda con el sentido de Berg,
    # sus ecuaciones con gamma positiva no representan ese sobrevuelo.
    valid_berg = bool(np.isclose(delta32, delta_2d_vectors, atol=0.5))
    return dict(alpha_deg=alpha, beta_deg=beta, up=u_planet, ui_berg=u_i,
                gamma_deg=geom["gamma_deg"], dU_32=delta32, dU_37=delta37,
                dU_vector_2d=delta_2d_vectors, dU_vector_3d=delta_3d,
                berg_orientation_valid=valid_berg,
                signed_rotation_indicator=signed)


def date_at(eph: Ephemeris, t: float | None) -> str:
    """Convierte tiempo absoluto a fecha TDB legible."""
    if t is None:
        return "no alcanzado"
    return str(eph.dates[0] + pd.to_timedelta(float(t-eph.t[0]), unit="s"))[:19] + " TDB"


def sensitivity_study(eph: dict[str, Ephemeris], phys: Physics, cfg: Settings,
                      t_target: float) -> pd.DataFrame:
    """Perturba el radio de empalme: mismo estado real de entrada por radio.

    En cada experimento el Modelo 2 arranca de esa MISMA condición. Se
    cuantifica diferencia de estados a la salida del Modelo 1 (mismo t),
    error respecto a Horizons y error proyectado al tiempo fijo t_target.
    """
    factors = sorted(set(cfg.sensitivity_factors+(phys.r_hill/phys.r_soi,)))
    rows: list[dict[str, Any]] = []
    for factor in factors:
        enc = encounter_real(eph, factor*phys.r_soi)
        t_pre = max(eph["jupiter"].t[0]+DAY, enc.t_in-20*DAY)
        orb1 = conics_orbit(eph,phys,cfg,enc,t_pre,t_target,name="Sensibilidad M1")
        orb2 = forced_orbit(eph,phys,cfg,enc,t_pre,t_target)
        tout = float(orb1.t_out)
        s1, s2 = orb1.state(tout),orb2.state(tout)
        truth_exit = eph["voyager"].state(tout)
        p1,p2 = orb1.state(t_target),orb2.state(t_target)
        truth = eph["voyager"].state(t_target)
        rows.append(dict(factor=factor,radius_km=enc.radius,
                         entry_jd=eph["jupiter"].jd[0]+enc.t_in/DAY,
                         exit_model1_jd=eph["jupiter"].jd[0]+tout/DAY,
                         delta_exit_r_km=np.linalg.norm(s1.r-s2.r),
                         delta_exit_v_kms=np.linalg.norm(s1.v-s2.v),
                         model1_exit_error_real_km=np.linalg.norm(s1.r-truth_exit.r),
                         model2_exit_error_real_km=np.linalg.norm(s2.r-truth_exit.r),
                         projected_delta_r_km=np.linalg.norm(p1.r-p2.r),
                         projected_delta_v_kms=np.linalg.norm(p1.v-p2.v),
                         projected_error_M1_real_km=np.linalg.norm(p1.r-truth.r),
                         projected_error_M2_real_km=np.linalg.norm(p2.r-truth.r)))
    return pd.DataFrame(rows)


def circular_jacobi_test(eph: dict[str, Ephemeris], phys: Physics,
                         cfg: Settings, encounter: Encounter) -> float:
    """Autoprueba: Jacobi casi exacta con planeta en movimiento circular uniforme."""
    t0 = encounter.t_peri
    J = eph["jupiter"].state(t0)
    axis = unit(np.cross(J.r,J.v))
    er = unit(J.r)
    et = unit(np.cross(axis,er))
    n = float(np.sqrt((phys.mu_sun+phys.mu_jupiter)/np.linalg.norm(J.r)**3))
    def circle_r(t: float) -> Array:
        theta=n*(t-t0)
        return np.linalg.norm(J.r)*(np.cos(theta)*er+np.sin(theta)*et)
    def rhs(t: float,y: Array) -> Array:
        dr=y[:3]-circle_r(t)
        rjc=circle_r(t)
        a=-phys.mu_sun*y[:3]/np.linalg.norm(y[:3])**3 - phys.mu_jupiter*(
            dr/np.linalg.norm(dr)**3 + rjc/np.linalg.norm(rjc)**3)
        return np.r_[y[3:],a]
    s0=eph["voyager"].state(t0)
    out=integration(rhs,t0,t0+75*DAY,s0.pack(),cfg)
    times=np.linspace(t0,t0+75*DAY,500)
    Y=out.sol(times).T
    rj=np.array([circle_r(t) for t in times])
    vj=np.array([n*np.cross(axis,p) for p in rj])
    C=jacobi_proxy(Y[:,:3],Y[:,3:],rj,vj,phys,n,axis)
    return float(np.max(np.abs(C-C[0]))/max(np.abs(C[0]),1.))


def make_plots(outdir: Path, eph: dict[str, Ephemeris], phys: Physics,
               cfg: Settings, enc: Encounter, models: list[Orbit],
               sens: pd.DataFrame | None, t_pre: float, t_end: float,
               t_target_real: float,
               geometry: dict[str,Any], berg: dict[str,float]) -> list[Path]:
    """Genera visualizaciones Matplotlib homogéneas para un informe."""
    plt.rcParams.update({"font.family":"DejaVu Sans", "font.size":10,
                         "axes.titlesize":12,"axes.labelsize":10,
                         "axes.grid":True,"grid.alpha":0.28,
                         "figure.figsize":(10,5.5), "legend.fontsize":8.5})
    colors=["#d95f02","#0072b2","#009e73"]
    saved=[]
    def save(fig: Any, name: str) -> None:
        fig.savefig(outdir/name,dpi=cfg.draw_dpi,bbox_inches="tight",facecolor="white")
        saved.append(outdir/name)
        plt.close(fig)
    def marks(ax: Any, before: bool=False) -> None:
        ax.axvline(0,color="gray",ls="--",lw=.9,alpha=.8)
        ax.axvline((enc.t_peri-enc.t_in)/DAY,color="#996515",ls=":",lw=1.1)
        ax.axvline((enc.t_out-enc.t_in)/DAY,color="gray",ls="--",lw=.9,alpha=.8)
    # Densidad temporal elevada en cercanías del periapsis.
    fine=np.linspace(enc.t_in,enc.t_out,950)
    near=np.linspace(enc.t_peri-3*DAY,enc.t_peri+3*DAY,850)
    all_t=np.unique(np.r_[np.linspace(t_pre,t_end,1200),fine,near])
    around=np.unique(np.r_[np.linspace(enc.t_in,enc.t_out,650),near])
    days=(around-enc.t_in)/DAY
    jr=eph["jupiter"].positions(around)
    jv=eph["jupiter"].velocities(around)
    vr=eph["voyager"].positions(around)
    vv=eph["voyager"].velocities(around)
    series=[m.states(around) for m in models]
    # 1: trayectoria jovicéntrica, proyección en plano orbital local.
    jp=eph["jupiter"].state(enc.t_peri)
    e1=unit(jp.r)
    e3=unit(np.cross(jp.r,jp.v))
    e2=np.cross(e3,e1)
    def xy(a: Array) -> tuple[Array,Array]:
        return (a@e2/1e6,a@e1/1e6)
    fig, axs=plt.subplots(1,2,figsize=(13.2,5.7))
    for ax in axs:
        ax.plot(*xy(vr-jr),color="black",lw=1.8,label="Horizons")
        for m,(r,v),c in zip(models,series,colors):
            ax.plot(*xy(r-jr),lw=1.3,color=c,label=m.name)
        ax.scatter([0],[0],color="#ad6728",s=80,zorder=5,label="Júpiter")
        ax.set_xlabel("Tangencial local (millones de km)")
        ax.set_ylabel("Radial local (millones de km)")
        ax.set_aspect("equal",adjustable="box")
    axs[0].set_title("Encuentro jovicéntrico (vista global)")
    axs[1].set_title("Detalle cercano a Júpiter")
    axs[1].set_xlim(-6,6);axs[1].set_ylim(-6,6)
    axs[0].legend(loc="best",framealpha=.95)
    save(fig,"01_trayectoria_jovicentrica.png")
    # 2 distancia SOI.
    fig,ax=plt.subplots()
    ax.plot(days,np.linalg.norm(vr-jr,axis=1)/1e6,color="black",lw=1.7,label="Horizons")
    for m,(r,v),c in zip(models,series,colors):
        ax.plot(days,np.linalg.norm(r-jr,axis=1)/1e6,color=c,label=m.name,lw=1.2)
    ax.axhline(phys.r_soi/1e6,color="gray",ls="--",label="SOI Laplace")
    ax.axhline(phys.r_hill/1e6,color="#9467bd",ls=":",label="Hill")
    marks(ax);ax.set(xlabel="Días desde entrada a SOI",ylabel="Distancia a Júpiter (10⁶ km)",title="Distancia a Júpiter y fronteras de empalme")
    ax.legend(ncol=2);save(fig,"02_distancia_jupiter.png")
    # 3 velocidades.
    fig,axs=plt.subplots(2,1,figsize=(10.8,8),sharex=True)
    axs[0].plot(days,np.linalg.norm(vv-jv,axis=1),color="black",label="Horizons")
    axs[1].plot(days,np.linalg.norm(vv,axis=1),color="black",label="Horizons")
    for m,(r,v),c in zip(models,series,colors):
        axs[0].plot(days,np.linalg.norm(v-jv,axis=1),color=c,label=m.name)
        axs[1].plot(days,np.linalg.norm(v,axis=1),color=c,label=m.name)
    for a in axs: marks(a);a.legend();a.set_ylabel("Rapidez (km/s)")
    axs[0].set_title("Rapidez jovicéntrica");axs[1].set_title("Rapidez heliocéntrica")
    axs[1].set_xlabel("Días desde entrada SOI")
    save(fig,"03_rapideces.png")
    # 4 invariantes: M1 solo dentro del intervalo real/hasta salida M1.
    fig,axs=plt.subplots(2,2,figsize=(13,8.6),sharex=True)
    days_ref=days
    for m,(r,v),c in zip(models,series,colors):
        rr=r-jr; relv=v-jv
        ej,hj=two_body_invariants(rr,relv,phys.mu_jupiter)
        es,hs=two_body_invariants(r,v,phys.mu_sun)
        mask=(around<=m.t_out) if m.t_out is not None else np.ones(len(around),dtype=bool)
        for ax,data in zip(axs.flat,[ej,np.linalg.norm(hj,axis=1),es,np.linalg.norm(hs,axis=1)]):
            ax.plot(days_ref[mask],data[mask],color=c,lw=1,label=m.name)
    esr,hsr=two_body_invariants(vr,vv,phys.mu_sun)
    ejr,hjr=two_body_invariants(vr-jr,vv-jv,phys.mu_jupiter)
    for ax,data in zip(axs.flat,[ejr,np.linalg.norm(hjr,axis=1),esr,np.linalg.norm(hsr,axis=1)]):
        ax.plot(days_ref,data,color="black",lw=1.5,alpha=.65,label="Horizons")
        marks(ax)
    for ax,title,y in zip(axs.flat,["Energía jovicéntrica εJ","Módulo de hJ","Energía heliocéntrica ε☉","Módulo de h☉"],
                          ["km²/s²","km²/s","km²/s²","km²/s"]):
        ax.set_title(title);ax.set_ylabel(y)
    axs[0,0].legend(fontsize=7.7)
    axs[1,0].set_xlabel("Días desde entrada SOI")
    axs[1,1].set_xlabel("Días desde entrada SOI")
    fig.suptitle("Conservación y transferencia de invariantes (valores absolutos)")
    save(fig,"04_invariantes_energia_momento.png")
    # 5 componentes del momento angular jovicéntrico, norma normalizada.
    fig,axs=plt.subplots(2,1,figsize=(10,7),sharex=True)
    for m,(r,v),c in zip(models,series,colors):
        hj=np.cross(r-jr,v-jv)
        msk=(around<=m.t_out) if m.t_out is not None else np.ones(len(around),bool)
        h0=np.linalg.norm(hj[0]);
        axs[0].plot(days[msk],np.linalg.norm(hj[msk],axis=1)/h0-1,color=c,label=m.name)
        axs[1].plot(days[msk],np.linalg.norm(hj[msk]-hj[0],axis=1)/h0,color=c,label=m.name)
    for a in axs:marks(a);a.legend()
    axs[0].set(ylabel="|hJ|/|hJ,entrada| − 1",title="Deriva relativa de la norma")
    axs[1].set(ylabel="|hJ(t)−hJ,entrada|/|hJ,entrada|",xlabel="Días desde entrada SOI",title="Cambio del vector completo")
    save(fig,"05_conservacion_vector_hj.png")
    # 6 marea y errores en la SOI (misma fecha).
    fig,axs=plt.subplots(3,1,figsize=(10.5,10),sharex=True)
    ratio=tidal_ratio(vr,jr,phys)
    axs[0].semilogy(days,ratio,color="black",label="Marea solar / fuerza jovicéntrica (real)")
    axs[0].axhline(0.1,color="#d95f02",ls=":",label="Referencia de 10 % (no umbral universal)")
    r1,v1=series[0]
    for m,(r,v),c in zip(models[1:],series[1:],colors[1:]):
        axs[1].semilogy(days,np.maximum(np.linalg.norm(r-r1,axis=1),1e-8),color=c,label=m.name)
        axs[2].semilogy(days,np.maximum(np.linalg.norm(v-v1,axis=1),1e-11),color=c,label=m.name)
    for ax in axs:marks(ax);ax.legend()
    axs[0].set_ylabel("Razón adimensional")
    axs[1].set_ylabel("Diferencia posición (km)")
    axs[2].set_ylabel("Diferencia velocidad (km/s)")
    axs[2].set_xlabel("Días desde entrada SOI")
    fig.suptitle("Validez de cónicas empalmadas: perturbación y divergencia")
    save(fig,"06_marea_y_error.png")
    # 7 cuasi Jacobi. Solo m1 dentro SOI para no sugerir conservación exterior.
    n=np.linalg.norm(np.cross(jp.r,jp.v))/np.linalg.norm(jp.r)**2
    axis=unit(np.cross(jp.r,jp.v))
    fig,ax=plt.subplots()
    for m,(r,v),c in zip(models,series,colors):
        cj=jacobi_proxy(r,v,jr,jv,phys,n,axis)
        mk=(around<=m.t_out) if m.t_out is not None else np.ones(len(around),dtype=bool)
        ax.plot(days[mk],cj[mk]-cj[0],color=c,label=m.name)
    cjreal=jacobi_proxy(vr,vv,jr,jv,phys,n,axis)
    ax.plot(days,cjreal-cjreal[0],color="black",label="Horizons",lw=1.4)
    marks(ax)
    ax.set(xlabel="Días desde entrada SOI",ylabel="C*(t) − C*(entrada) (km²/s²)",
           title="Cuasi-Jacobi: diagnóstico, NO constante exacta con órbitas reales")
    ax.legend();save(fig,"07_cuasi_jacobi.png")
    # 8 Trayectoria heliocéntrica hasta el radio objetivo (sin Saturno dinámico).
    fig,ax=plt.subplots(figsize=(9.3,9.3))
    theta=np.linspace(0,2*np.pi,600)
    for radius,label,ls in [(phys.a_jupiter/AU,"Órbita media de Júpiter",":"),
                            (phys.target_radius_au,"Radio objetivo 9.58 UA (referencia)","--")]:
        ax.plot(radius*np.cos(theta),radius*np.sin(theta),ls,color="gray",lw=1,label=label)
    plot_t=all_t[all_t<=min(t_end,t_target_real+50*DAY)]
    xreal=eph["voyager"].positions(plot_t)/AU
    ax.plot(xreal[:,0],xreal[:,1],color="black",lw=1.8,label="Voyager real")
    for m,c in zip(models,colors):
        r,_=m.states(plot_t)
        ax.plot(r[:,0]/AU,r[:,1]/AU,color=c,lw=1.5,label=m.name)
    rjperi=jp.r/AU
    ax.scatter(*rjperi[:2],color="#a87926",s=75,marker="o",label="Júpiter (periapsis real)")
    ax.scatter([0],[0],color="#f0b429",s=115,zorder=5,label="Sol")
    ax.set(xlabel="X heliocéntrica (UA)",ylabel="Y heliocéntrica (UA)",
           title="Voyager 2: trayectoria heliocéntrica y radio objetivo")
    ax.set_aspect("equal",adjustable="box");ax.legend(loc="best")
    save(fig,"08_trayectoria_radio_objetivo.png")
    # 9 Radio heliocéntrico: prueba geométrica sin fuerzas de Saturno.
    dts=np.linspace(max(enc.t_out,t_target_real-350*DAY),
                    min(t_end,t_target_real+120*DAY),520)
    fig,axs=plt.subplots(2,1,figsize=(11.5,8.0),sharex=True)
    axs[0].plot((dts-t_target_real)/DAY,
                np.linalg.norm(eph["voyager"].positions(dts),axis=1)/AU,
                color="black",lw=1.8,label="Horizons")
    truth=eph["voyager"].positions(dts)
    for m,c in zip(models,colors):
        r,_=m.states(dts)
        axs[0].plot((dts-t_target_real)/DAY,np.linalg.norm(r,axis=1)/AU,
                    color=c,label=m.name)
        axs[1].plot((dts-t_target_real)/DAY,
                    np.linalg.norm(r-truth,axis=1)/1e6,color=c,label=m.name)
    axs[0].axhline(phys.target_radius_au,color="gray",ls="--",label="9.58 UA: radio objetivo")
    axs[0].set_ylabel("Distancia al Sol (UA)")
    axs[1].set_ylabel("Error posicional contra Horizons (10⁶ km)")
    axs[1].set_xlabel("Días respecto al cruce real del radio de 9.58 UA")
    axs[0].legend();axs[1].legend()
    fig.suptitle("Cruce geométrico del radio objetivo, sin dinámica de Saturno")
    save(fig,"09_radio_objetivo_y_error.png")
    # 10 sensibilidad de empalme.
    if sens is not None:
        fig,axs=plt.subplots(1,2,figsize=(12.8,4.8))
        xs=sens.factor
        axs[0].plot(xs,sens.delta_exit_r_km/1e6,"o-",label="M1 vs M2, igual fecha")
        axs[0].plot(xs,sens.model1_exit_error_real_km/1e6,"s--",label="M1 vs Horizons")
        axs[1].plot(xs,sens.projected_delta_r_km/1e6,"o-",label="M1 vs M2, fecha objetivo")
        axs[1].plot(xs,sens.projected_error_M1_real_km/1e6,"s--",label="M1 vs Horizons")
        for ax in axs:
            ax.axvline(1,color="gray",ls=":")
            ax.set_xlabel("Radio de empalme / R_SOI")
            ax.set_ylabel("Error de posición (10⁶ km)")
            ax.legend()
        axs[0].set_title("Error a salida del Modelo 1")
        axs[1].set_title("Error proyectado hasta 9.58 UA")
        fig.suptitle("Sensibilidad a radio de empalme (M2: Sol+Júpiter)")
        save(fig,"10_sensibilidad_soi.png")
    # 11 comparación rapidez: Berg, vectores 3D, modelos y efemérides.
    fig,ax=plt.subplots(figsize=(9.5,5.2))
    s_in=eph["voyager"].state(enc.t_in)
    s_real_out=eph["voyager"].state(enc.t_out)
    values=[berg["dU_32"],berg["dU_37"],berg["dU_vector_3d"],
            np.linalg.norm(s_real_out.v)-np.linalg.norm(s_in.v)]
    labels=["Berg (32)","Berg (37)","Asintótica 3D", "Horizons (bordes SOI)"]
    for m in models:
        labels.append(m.name.replace("Modelo ","M"))
        values.append(np.linalg.norm(m.state(m.t_out if m.t_out else enc.t_out).v)-np.linalg.norm(s_in.v))
    ax.barh(labels,values,color=["#b07aa1","#b07aa1","#6b6ecf","#262626"]+colors)
    ax.axvline(0,color="gray",lw=1)
    ax.set(xlabel="Ganancia heliocéntrica de rapidez (km/s)",
           title="Ganancia analítica y numérica (definiciones distintas)")
    for i,v in enumerate(values):ax.text(v+.12,i,f"{v:.2f}",va="center",fontsize=8)
    fig.tight_layout();save(fig,"11_validacion_berg.png")
    return saved


def plotly_animation_2d(outdir: Path, eph: dict[str, Ephemeris],
                        models: list[Orbit], encounter: Encounter) -> Path | None:
    """HTML animado: estado relativo y energía en un instante durante sobrevuelo.

    Reemplaza la animación de Colab por una exportación reproducible,
    con controles de reproducción, pausa y fechas TDB.
    """
    try:
        import plotly.graph_objects as go
    except ImportError:
        return None
    times=np.linspace(encounter.t_in,encounter.t_out,120)
    rrj=eph["jupiter"].positions(times)
    vr=eph["voyager"].positions(times)-rrj
    data=[m.states(times) for m in models]
    fig=go.Figure()
    palette=["#d95f02","#0072b2","#009e73"]
    fig.add_trace(go.Scatter(x=vr[:,0]/1e6,y=vr[:,1]/1e6,mode="lines",
                             line=dict(color="black",width=2),name="Horizons"))
    for m,(r,v),c in zip(models,data,palette):
        rel=(r-rrj)/1e6
        fig.add_trace(go.Scatter(x=rel[:,0],y=rel[:,1],mode="lines",
                                 line=dict(color=c,width=2),name=m.name))
    fig.add_trace(go.Scatter(x=[0],y=[0],mode="markers",marker=dict(size=13,color="#bb762d"),name="Júpiter"))
    fig.add_trace(go.Scatter(x=[vr[0,0]/1e6],y=[vr[0,1]/1e6],mode="markers",
                             marker=dict(size=12,color="#b13045",symbol="diamond"),name="Voyager (real)"))
    frames=[]
    steps=[]
    for k,t in enumerate(times):
        label=date_at(eph["jupiter"],t)[:16]
        frames.append(go.Frame(name=str(k),traces=[len(models)+2],
                    data=[go.Scatter(x=[vr[k,0]/1e6],y=[vr[k,1]/1e6])],
                    layout=go.Layout(title=f"Encuentro Voyager 2-Júpiter | {label} TDB")))
        steps.append(dict(method="animate",label=str(k),args=[[str(k)],dict(mode="immediate",
                frame=dict(duration=0,redraw=False),transition=dict(duration=0))]))
    fig.frames=frames
    fig.update_layout(title="Encuentro Voyager 2-Júpiter: evolución 2D",
        xaxis_title="X jovicéntrica eclíptica (10⁶ km)",
        yaxis_title="Y jovicéntrica eclíptica (10⁶ km)",
        xaxis=dict(scaleanchor="y",scaleratio=1),template="plotly_white",
        updatemenus=[dict(type="buttons",showactive=False,buttons=[
            dict(label="Reproducir",method="animate",args=[None,dict(frame=dict(duration=90,redraw=False),fromcurrent=True)]),
            dict(label="Pausa",method="animate",args=[[None],dict(mode="immediate",frame=dict(duration=0,redraw=False))])])],
        sliders=[dict(active=0,steps=steps,currentvalue=dict(prefix="Fotograma "))],
        width=980,height=700)
    path=outdir/"13_animacion_encuentro_2d.html"
    fig.write_html(path,include_plotlyjs=True,auto_open=False)
    return path


def main() -> None:
    """Orquesta la comparación, validaciones físicas, tablas y figuras."""
    ap=argparse.ArgumentParser(description=__doc__,formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("--jupiter",type=Path,default=Path("horizons_jupiter.csv"))
    ap.add_argument("--voyager",type=Path,default=Path("horizons_voyager-2.csv"))
    ap.add_argument("--sin-sensibilidad",action="store_true")
    ap.add_argument("--sin-plotly",action="store_true")
    ap.add_argument("--salida",type=Path,default=Path("resultados_voyager2_reto_III3"))
    args=ap.parse_args()
    args.salida.mkdir(parents=True,exist_ok=True)
    phys,cfg=Physics(),Settings()
    paths={"jupiter":args.jupiter,"voyager":args.voyager}
    eph=synchronize_data(paths)
    enc=encounter_real(eph,phys.r_soi)
    target_radius=phys.target_radius_au*AU
    t_target_real=find_crossing(eph["voyager"],target_radius,enc.t_out,eph["voyager"].t[-1])
    if t_target_real is None:
        raise RuntimeError("Voyager real no alcanzó 9.58 UA en el intervalo de Horizons")
    t_pre=max(eph["voyager"].t[0]+DAY,enc.t_in-cfg.days_before_entry*DAY)
    t_end=min(eph["voyager"].t[-1]-DAY,t_target_real+cfg.days_after_target_cross*DAY)
    print("[1/5] Propagando dos modelos con Sol, Júpiter y sonda; sin Saturno...")
    m1=conics_orbit(eph,phys,cfg,enc,t_pre,t_end)
    m2=forced_orbit(eph,phys,cfg,enc,t_pre,t_end)
    models=[m1,m2]
    s_in=eph["voyager"].state(enc.t_in)
    s_out=eph["voyager"].state(enc.t_out)
    jov=eph["jupiter"].state(enc.t_in)
    geom=hyperbola_geometry(State(s_in.r-jov.r,s_in.v-jov.v),phys)
    berg=berg_parameters(geom,jov)
    # El análisis reporta siempre cuál es la salida común usada.
    main_rows=[]
    elements_rows=[]
    for model_name,st_in,st_out in [("Horizons",s_in,s_out)]+[
        (m.name,m.state(enc.t_in),m.state(m.t_out if m.t_out is not None else enc.t_out)) for m in models]:
        el_in,el_out=orbit_elements(st_in,phys.mu_sun),orbit_elements(st_out,phys.mu_sun)
        for when,e in [("antes_SOI",el_in),("despues_SOI",el_out)]:
            elements_rows.append(dict(modelo=model_name,instante=when,
                                      a_UA=e.a_au,e=e.eccentricity,
                                      afelio_UA=e.aphelion_au,
                                      inclinacion_grados=e.inclination_deg,
                                      energia_solar_km2_s2=e.energy))
    print("[2/5] Validando invariantes y cruce del radio geométrico de 9.58 UA...")
    for m in models:
        t_cross=first_radius_time(m,enc.t_out,t_end,target_radius)
        at_target=m.state(t_target_real)
        real_target=eph["voyager"].state(t_target_real)
        t_exit=m.t_out if m.t_out is not None else enc.t_out
        exit_state=m.state(t_exit)
        real_exit_same_date=eph["voyager"].state(t_exit)
        row=dict(model=m.name,salida_fecha=date_at(eph["jupiter"],t_exit),
                 salida_JD=eph["jupiter"].jd[0]+t_exit/DAY,
                 delta_energia_solar_km2_s2=orbit_elements(exit_state,phys.mu_sun).energy-orbit_elements(s_in,phys.mu_sun).energy,
                 delta_rapidez_solar_kms=np.linalg.norm(exit_state.v)-np.linalg.norm(s_in.v),
                 error_salida_r_km=np.linalg.norm(exit_state.r-real_exit_same_date.r),
                 error_salida_v_kms=np.linalg.norm(exit_state.v-real_exit_same_date.v),
                 cruce_radio_9p58_UA=date_at(eph["jupiter"],t_cross),
                 cruce_radio_9p58_JD=eph["jupiter"].jd[0]+t_cross/DAY if t_cross is not None else np.nan,
                 error_posicion_fecha_cruce_real_km=np.linalg.norm(at_target.r-real_target.r),
                 error_velocidad_fecha_cruce_real_kms=np.linalg.norm(at_target.v-real_target.v))
        main_rows.append(row)
    results=pd.DataFrame(main_rows)
    elements=pd.DataFrame(elements_rows)
    # Comprobación de conservación rigurosa en M1, y deriva jovicéntrica M2.
    t_inside=np.linspace(enc.t_in,min(enc.t_out,float(m1.t_out)),1100)
    jj=eph["jupiter"].positions(t_inside)
    jv=eph["jupiter"].velocities(t_inside)
    invariant_summary={}
    for m in models:
        r,v=m.states(t_inside)
        ej,hj=two_body_invariants(r-jj,v-jv,phys.mu_jupiter)
        invariant_summary[m.name]={"max_abs_delta_epsJ_km2_s2":float(np.ptp(ej)),
            "max_relative_norm_hJ":float(np.max(np.abs(np.linalg.norm(hj,axis=1)/np.linalg.norm(hj[0])-1))),
            "max_relative_vector_hJ":float(np.max(np.linalg.norm(hj-hj[0],axis=1)/np.linalg.norm(hj[0])))}
    t_peri=eph["jupiter"].state(enc.t_peri)
    axis=unit(np.cross(t_peri.r,t_peri.v))
    n=np.linalg.norm(np.cross(t_peri.r,t_peri.v))/np.linalg.norm(t_peri.r)**2
    jacobi_summary={}
    for m in models:
        r,v=m.states(t_inside)
        C=jacobi_proxy(r,v,jj,jv,phys,n,axis)
        jacobi_summary[m.name]={"C_inicio":float(C[0]),"delta_C_max":float(np.max(np.abs(C-C[0])))}
    jacobi_test=circular_jacobi_test(eph,phys,cfg,enc)
    # La divergencia no tiene un umbral universal. Reportar hitos descriptivos.
    ra1,va1=m1.states(t_inside)
    ra2,va2=m2.states(t_inside)
    diff_r=np.linalg.norm(ra1-ra2,axis=1)
    diff_v=np.linalg.norm(va1-va2,axis=1)
    def first_error(threshold: float, values: Array) -> str:
        where=np.flatnonzero(values>=threshold)
        return date_at(eph["jupiter"],float(t_inside[where[0]])) if len(where) else "no alcanzado"
    divergence_diagnostic={
        "primer_error_posicion_superior_a_1000_km":first_error(1000.0,diff_r),
        "primer_error_posicion_superior_a_100000_km":first_error(100000.0,diff_r),
        "primer_error_velocidad_superior_a_0p1_kms":first_error(0.1,diff_v),
        "nota":"Umbrales ilustrativos, NO frontera física universal de validez"}
    print("[3/5] Ejecutando estudio de sensibilidad de frontera...")
    sens=None if args.sin_sensibilidad else sensitivity_study(eph,phys,cfg,t_target_real)
    print("[4/5] Generando figuras científicas...")
    figures=make_plots(args.salida,eph,phys,cfg,enc,models,sens,t_pre,t_end,
                       t_target_real,geom,berg)
    if not args.sin_plotly:
        animated=plotly_animation_2d(args.salida,eph,models,enc)
        if animated is not None:figures.append(animated)
    # Exportaciones: todos los resultados reproducibles.
    results.to_csv(args.salida/"comparacion_modelos.csv",index=False,encoding="utf-8-sig")
    elements.to_csv(args.salida/"elementos_osculatrices.csv",index=False,encoding="utf-8-sig")
    if sens is not None:
        sens.to_csv(args.salida/"sensibilidad_radio_soi.csv",index=False,encoding="utf-8-sig")
    tdiag=np.unique(np.r_[np.linspace(enc.t_in,enc.t_out,600),
                           np.linspace(enc.t_peri-DAY,enc.t_peri+DAY,300)])
    rj=eph["jupiter"].positions(tdiag)
    rv=eph["voyager"].positions(tdiag)
    ratio=tidal_ratio(rv,rj,phys)
    diag=pd.DataFrame({"JD":eph["jupiter"].jd[0]+tdiag/DAY,
                       "dias_desde_entrada_SOI":(tdiag-enc.t_in)/DAY,
                       "razon_marea_solar_a_jupiter":ratio})
    for k,m in enumerate(models):
        rr,vv=m.states(tdiag)
        eJ,hJ=two_body_invariants(rr-rj,vv-eph["jupiter"].velocities(tdiag),phys.mu_jupiter)
        es,hs=two_body_invariants(rr,vv,phys.mu_sun)
        diag[f"M{k+1}_energia_J"]=eJ
        diag[f"M{k+1}_hJ_x"]=hJ[:,0]
        diag[f"M{k+1}_hJ_y"]=hJ[:,1]
        diag[f"M{k+1}_hJ_z"]=hJ[:,2]
        diag[f"M{k+1}_energia_Sol"]=es
        diag[f"M{k+1}_h_Sol_norm"]=np.linalg.norm(hs,axis=1)
    diag.to_csv(args.salida/"diagnosticos_temporales.csv",index=False,encoding="utf-8-sig")
    report=dict(marco="Heliocéntrico, eclíptica J2000, KM-S, TDB",
        modelo2="Sonda restringida Sol+Júpiter, efemérides prescritas de Júpiter; no autónomo",
        laplace_km=phys.r_soi,hill_km=phys.r_hill,
        encuentro_real=dict(entrada=date_at(eph["jupiter"],enc.t_in),
            periapsis=date_at(eph["jupiter"],enc.t_peri),
            salida=date_at(eph["jupiter"],enc.t_out),distancia_minima_km=enc.r_peri),
        objetivo_geometrico=dict(radio_ua=phys.target_radius_au,
            cruce_real_9_58_ua=date_at(eph["jupiter"],t_target_real),
            comentario="Radio fijo de referencia; no hay efemérides ni gravedad de Saturno"),
        hiperbola={k:v for k,v in geom.items() if not isinstance(v,np.ndarray)},
        berg=berg,conservacion=invariant_summary,divergencia=divergence_diagnostic,
        cuasi_jacobi=jacobi_summary,
        autoprueba_jacobi_circular_deriva_relativa=jacobi_test,
        razon_marea_min=float(np.min(ratio)),razon_marea_max=float(np.max(ratio)),
        archivos_graficas=[p.name for p in figures])
    (args.salida/"resumen_fisico.json").write_text(
        json.dumps(report,indent=2,ensure_ascii=False,allow_nan=False),encoding="utf-8")
    print("\n=== DATOS DEL ENCUENTRO (Horizons) ===")
    print(f"Laplace: {phys.r_soi:,.0f} km; Hill: {phys.r_hill:,.0f} km")
    print(f"Entrada: {date_at(eph['jupiter'],enc.t_in)}")
    print(f"Periapsis: {date_at(eph['jupiter'],enc.t_peri)} a {enc.r_peri:,.0f} km")
    print(f"Salida: {date_at(eph['jupiter'],enc.t_out)}")
    print(f"Radio objetivo geométrico: {phys.target_radius_au:.2f} UA, sin modelar Saturno")
    print(f"Berg: b={geom['b']:,.0f} km, D={geom['D']:.5f}, γ={geom['gamma_deg']:.3f}°")
    print(f"α={berg['alpha_deg']:.3f}°, β={berg['beta_deg']:.3f}°; "
          f"Δu (32)={berg['dU_32']:.4f}, Δu (37)={berg['dU_37']:.4f} km/s")
    print(f"Autoprueba Jacobi (caso circular ideal): deriva relativa máx. {jacobi_test:.3e}")
    print("\n=== ELEMENTOS HELIOCÉNTRICOS ===\n",elements.to_string(index=False))
    print("\n=== COMPARACIÓN MODELOS ===\n",results.to_string(index=False))
    print("\n=== CONSERVACIÓN ===\n",json.dumps(invariant_summary,indent=2,ensure_ascii=False))
    print("\n=== HITOS DE DIVERGENCIA ===\n",json.dumps(divergence_diagnostic,indent=2,ensure_ascii=False))
    print(f"\n[5/5] Archivos exportados en {args.salida.resolve()}")


## 3. Ejecutar el estudio y generar las figuras

Se generan 11 figuras PNG, la animación 2D que funciona sin WebGL, las tablas CSV de comparación y sensibilidad, y un resumen físico JSON. Los resultados se guardan en `/content/resultados_voyager2_sol_jupiter`. Cada ejecución del análisis conserva los archivos válidos de esta versión.

In [4]:
# 3. Ejecutar la simulación completa: dos modelos, dos CSV, sin Saturno
import sys
from pathlib import Path
SALIDA = Path('/content/resultados_voyager2_sol_jupiter')
SALIDA.mkdir(parents=True, exist_ok=True)
args_anteriores = sys.argv[:]
try:
    sys.argv = ['asistencia_gravitacional_sol_jupiter.py',
                '--jupiter', '/content/horizons_jupiter.csv',
                '--voyager', '/content/horizons_voyager-2.csv',
                '--salida', str(SALIDA)]
    main()
finally:
    sys.argv = args_anteriores
print('Resultados científicos guardados en', SALIDA)

[1/5] Propagando dos modelos con Sol, Júpiter y sonda; sin Saturno...
[2/5] Validando invariantes y cruce del radio geométrico de 9.58 UA...
[3/5] Ejecutando estudio de sensibilidad de frontera...
[4/5] Generando figuras científicas...

=== DATOS DEL ENCUENTRO (Horizons) ===
Laplace: 48,219,248 km; Hill: 53,152,809 km
Entrada: 1979-05-06 11:34:09 TDB
Periapsis: 1979-07-09 22:26:54 TDB a 720,894 km
Salida: 1979-09-12 20:40:14 TDB
Radio objetivo geométrico: 9.58 UA, sin modelar Saturno
Berg: b=1,654,970 km, D=1.30015, γ=104.869°
α=53.337°, β=51.499°; Δu (32)=10.0076, Δu (37)=10.0076 km/s
Autoprueba Jacobi (caso circular ideal): deriva relativa máx. 1.556e-10

=== ELEMENTOS HELIOCÉNTRICOS ===
                modelo    instante       a_UA        e  afelio_UA  inclinacion_grados  energia_solar_km2_s2
             Horizons   antes_SOI   3.652355 0.737106   6.344529            5.120500           -121.446001
             Horizons despues_SOI -14.740788 1.340552        inf            2.579877  

## 4. Visualización tridimensional libre sin WebGL

La simulación utiliza **Canvas 2D con proyección de coordenadas 3D**, sin dependencias de WebGL. **Centrado inicial en Júpiter**. Arrastra para girar, **Mayús + arrastrar** o botón derecho para desplazarte por el espacio y usa la rueda para acercar/alejar. Puedes seguir a Júpiter, Voyager o el Sol, escoger entre Horizons/Modelo 1/Modelo 2, y leer posición, velocidad, energías, momento angular y error en cada instante. Las gráficas de la parte inferior se actualizan con la animación.

In [5]:
# 4. Ajustes de los dos HTML (sin tocar las ecuaciones)
CUADROS_3D = 260          # muestras de animación, se añaden más cerca del periapsis
PERIODO_MS = 85           # milisegundos entre fotogramas
print('Cuadros:', CUADROS_3D, 'periodo (ms):', PERIODO_MS)

Cuadros: 260 periodo (ms): 85


In [6]:
def generar_panel3d(escena: str="jovicentrica", mostrar: bool=True):
    # ================================================================
    # VISUALIZACIÓN INTERACTIVA CON CÁMARA LIBRE, CENTRADA EN JÚPITER
    # Pegar como UNA CELDA debajo del código científico del cuaderno.
    # Compatible con Google Colab: la animación usa Canvas 2D (no Scatter3d).
    # ================================================================
    from pathlib import Path
    from IPython.display import HTML, display
    import html
    import json
    import numpy as np
    import pandas as pd

    # CONFIGURACIÓN
    ESCENA = escena  # "heliocentrica" o "jovicentrica"
    CUADROS_3D = int(globals().get("CUADROS_3D", 260))  # Fotogramas
    PERIODO_MS = int(globals().get("PERIODO_MS", 90))  # Milisegundos/fotograma
    ARCHIVO_HTML = ("12_trayectoria_heliocentrica_sin_saturno.html"
                    if ESCENA == "heliocentrica" else "13_sobrevuelo_jupiter_centrado.html")

    # Usa las funciones que ya ejecutaste en el notebook. Si el archivo .py
    # está subido a Colab, también puede importarlas desde allí.
    _nombres = ("Physics", "Settings", "synchronize_data", "encounter_real",
                "conics_orbit", "forced_orbit", "date_at", "DAY")
    if not all(x in globals() for x in _nombres):
        raise RuntimeError("Primero ejecuta la celda científica (sección 2).")

    # Localiza los CSV sin pedir que vuelvas a cargarlos.
    def _buscar_csv(nombre):
        for raiz in (Path("/content"), Path.cwd(), Path("/mnt/data")):
            ruta = raiz / nombre
            if ruta.is_file():
                return ruta
        raise FileNotFoundError(
            f"No aparece {nombre}. Súbelo desde el icono de carpeta en Colab."
        )

    _csv = {"jupiter": _buscar_csv("horizons_jupiter.csv"),
            "voyager": _buscar_csv("horizons_voyager-2.csv")}
    _phys, _config = Physics(), Settings()
    _eph = synchronize_data(_csv)
    _enc = encounter_real(_eph, _phys.r_soi)
    if ESCENA not in ("heliocentrica", "jovicentrica"):
        raise ValueError("ESCENA debe ser 'heliocentrica' o 'jovicentrica'")
    _t0 = float(_enc.t_in)
    if ESCENA == "heliocentrica":
        # Propagamos a un radio comparable con 9.58 UA sin utilizar gravedad de Saturno.
        _pre = max(float(_eph["jupiter"].t[0]) + DAY, _t0-100*DAY)
        _tf = min(float(_eph["jupiter"].t[-1]) - DAY, _t0+1000*DAY)
    else:
        _pre = _t0 - DAY
        _tf = float(_enc.t_out)
    _m1 = conics_orbit(_eph, _phys, _config, _enc, _pre, _tf+10*DAY)
    _m2 = forced_orbit(_eph, _phys, _config, _enc, _pre, _tf+10*DAY)

    # En heliocéntrico empieza antes del encuentro; en jovicéntrico empieza al entrar a SOI.
    # Malla densa cerca del periapsis y otra para cubrir la trayectoria completa.
    # El muestreo no uniforme evita perder el pico de rapidez/energía.
    _t_inicial = _pre if ESCENA == "heliocentrica" else _t0
    _tiempos = np.unique(np.r_[np.linspace(_t_inicial, _tf, CUADROS_3D),
        np.linspace(max(_t_inicial, _enc.t_peri-30*DAY),
                    min(_tf, _enc.t_peri+30*DAY), 170),
        np.linspace(max(_t_inicial, _enc.t_peri-3*DAY),
                    min(_tf, _enc.t_peri+3*DAY), 120),
        _enc.t_in, _enc.t_peri, _enc.t_out])
    _tiempos = _tiempos[(_tiempos >= _t_inicial) & (_tiempos <= _tf)]
    _rj = _eph["jupiter"].positions(_tiempos)
    _vj = _eph["jupiter"].velocities(_tiempos)
    _rh = _eph["voyager"].positions(_tiempos)
    _vh = _eph["voyager"].velocities(_tiempos)
    _r1, _v1 = _m1.states(_tiempos)
    _r2, _v2 = _m2.states(_tiempos)

    # Tanto en vistas jovicéntricas como heliocéntricas se representan datos 3D reales.
    # Posiciones: millones de km (encuentro) o UA (trayectoria solar).
    _AU = 149597870.7
    _escala = _AU if ESCENA == "heliocentrica" else 1e6
    _origen = np.zeros_like(_rj) if ESCENA == "heliocentrica" else _rj
    _series = {
        "Horizons": ((_rh-_origen)/_escala, _vh-_vj, _rh-_rj),
        "Modelo 1": ((_r1-_origen)/_escala, _v1-_vj, _r1-_rj),
        "Modelo 2": ((_r2-_origen)/_escala, _v2-_vj, _r2-_rj),
    }
    # Las magnitudes específicas son por unidad de masa de la sonda.
    # Se calcula cada serie con SU PROPIA posición y velocidad heliocéntricas.
    # K_sun = v²/2, U_sun = -mu_sun/r, eps_sun = K_sun + U_sun.
    # K_j = |v-vJ|²/2, U_j = -mu_j/|r-rJ|: solo problema jovicéntrico.
    # h_sun = |r x v|, h_j = |(r-rJ) x (v-vJ)|.
    # En el modelo forzado ni eps_sun ni eps_j son invariantes exactos.
    _helio = {"Horizons": (_rh, _vh), "Modelo 1": (_r1, _v1),
              "Modelo 2": (_r2, _v2)}
    _data = {}
    _for_name = ["Horizons", "Modelo 1", "Modelo 2"]
    for _nombre in _for_name:
        _rH, _vH = _helio[_nombre]
        _r_rel, _v_rel = _rH - _rj, _vH - _vj
        _r_draw = (_rH - _origen) / _escala
        _dJ = np.linalg.norm(_r_rel, axis=1)
        _dSol = np.linalg.norm(_rH, axis=1)
        _sH = np.linalg.norm(_vH, axis=1)
        _sJ = np.linalg.norm(_v_rel, axis=1)
        _Ksun, _Usun = 0.5*_sH**2, -_phys.mu_sun/_dSol
        _Kj, _Uj = 0.5*_sJ**2, -_phys.mu_jupiter/_dJ
        _hsun = np.linalg.norm(np.cross(_rH, _vH), axis=1)
        _hj = np.linalg.norm(np.cross(_r_rel, _v_rel), axis=1)
        _error_pos = np.linalg.norm(_rH-_rh, axis=1)
        _error_vel = np.linalg.norm(_vH-_vh, axis=1)
        _payload_one = {
           "pos": _r_draw, "rHelio": _rH/_AU, "rJov": _r_rel/1e6,
           "vHelio": _vH, "vJov": _v_rel, "vel": _v_rel,
           "rSol": _dSol/_AU, "dJ": _dJ/1e6,
           "restante": 9.58-_dSol/_AU, "rapH": _sH, "rapJ": _sJ,
           "Ksun": _Ksun, "Usun": _Usun, "Esun": _Ksun+_Usun,
           "Kj": _Kj, "Uj": _Uj, "Ej": _Kj+_Uj,
           "hSun": _hsun/1e9, "hJ": _hj/1e6,
           "errPos": _error_pos/1e6, "errVel": _error_vel
        }
        _data[_nombre] = {
            _key: None if _arr is None else np.round(_arr, 8).tolist()
            for _key, _arr in _payload_one.items()
        }

    _jupiter_3d = np.round(_rj/_escala, 8).tolist() if ESCENA == "heliocentrica" else [[0.0, 0.0, 0.0] for _ in range(len(_tiempos))]
    _dist_real = np.linalg.norm(_rh-_rj, axis=1)
    _peri_ind = int(np.argmin(np.abs(_tiempos-_enc.t_peri)))
    _max_radio = max(float(np.max(np.linalg.norm(_x, axis=1)))
                     for _x in (_data[_n]["pos"] for _n in _for_name))
    _payload = {
        "mode": ESCENA,
        "series": _data,
        "dates": [date_at(_eph["jupiter"], float(tt))[:19] for tt in _tiempos],
        "days": np.round((_tiempos-_enc.t_in)/DAY, 8).tolist(),
        "periDay": float((_enc.t_peri-_enc.t_in)/DAY),
        "outDay": float((_enc.t_out-_enc.t_in)/DAY),
        "soi": round(_phys.r_soi / _escala, 8),
        "peri": _peri_ind,
        "periDist": float(_dist_real[_peri_ind]/_escala),
        "planetJ": _jupiter_3d,
        "planetSun": [[0.0, 0.0, 0.0] for _ in range(len(_tiempos))] if ESCENA == "heliocentrica" else np.round(-_rj/_escala, 8).tolist(),
        "fullSpan": float(_max_radio * 1.22),
        "units": "UA" if ESCENA == "heliocentrica" else "10⁶ km",
        "frameMs": int(PERIODO_MS),
    }
    _payload_json = json.dumps(_payload, ensure_ascii=False, separators=(",", ":"),allow_nan=False)
    _payload_json = _payload_json.replace("</", "<\\/")

    # Datos de la animación para reproducibilidad y tablas propias.
    # Cada fila es una fecha, para uno de los tres modelos.
    _columnas = {
        "rHelio": ["x_heliocentrico_UA", "y_heliocentrico_UA", "z_heliocentrico_UA"],
        "rJov": ["x_jovicentrico_millones_km", "y_jovicentrico_millones_km", "z_jovicentrico_millones_km"],
        "vHelio": ["vx_heliocentrico_km_s", "vy_heliocentrico_km_s", "vz_heliocentrico_km_s"],
        "vJov": ["vx_jovicentrico_km_s", "vy_jovicentrico_km_s", "vz_jovicentrico_km_s"]
    }
    _tablas = []
    for _name in _for_name:
        _series_modelo = _data[_name]
        _csv_df = pd.DataFrame({
            "modelo": _name, "fecha_TDB": _payload["dates"],
            "dias_desde_entrada_SOI": _payload["days"],
            **{key: _series_modelo[key] for key in (
                "rSol", "dJ", "rapH", "rapJ", "Ksun", "Usun", "Esun",
                "Kj", "Uj", "Ej", "hSun", "hJ", "restante", "errPos", "errVel")},
        })
        for _key, _fields in _columnas.items():
            _val = np.asarray(_series_modelo[_key])
            for _axis, _field in enumerate(_fields):
                _csv_df[_field] = _val[:, _axis]
        _tablas.append(_csv_df)
    _result_dir = ((Path("/content") if Path("/content").exists() else Path.cwd()) /
                   "resultados_voyager2_sol_jupiter")
    _result_dir.mkdir(parents=True, exist_ok=True)
    _tele_csv = _result_dir / f"telemetria_dashboard_{ESCENA}.csv"
    pd.concat(_tablas, ignore_index=True).to_csv(_tele_csv, index=False, encoding="utf-8-sig")
    print("Telemetría exportada:", _tele_csv)

    # 3D REAL POR PROYECCIÓN: Canvas 2D no necesita WebGL ni GPU.
    # Toda la lógica de cámara, rotación y controles vive dentro del HTML.
    _documento = r'''<!DOCTYPE html>
    <html lang="es"><head><meta charset="utf-8">
    <meta name="viewport" content="width=device-width,initial-scale=1">
    <title>Voyager 2 - Jupiter | 3D sin WebGL</title>
    <style>
    *{box-sizing:border-box}body{margin:0;background:#0b1020;color:#e6edff;font-family:system-ui,-apple-system,Segoe UI,sans-serif}
    #v2{max-width:1350px;margin:0 auto;padding:13px}
    header{display:flex;gap:12px;align-items:center;justify-content:space-between;flex-wrap:wrap;margin:0 0 10px}
    h1{font-size:19px;margin:0;font-weight:700}small{color:#96a6c6}
    .controls,.timeline{display:flex;gap:9px;align-items:center;flex-wrap:wrap}
    button,select{background:#182643;color:#edf3ff;border:1px solid #344668;padding:9px 12px;border-radius:9px;cursor:pointer;font-size:13px}
    button:hover{background:#263c61}button:focus,select:focus{outline:2px solid #53c5fc}
    .scene{position:relative;height:530px;border:1px solid #30405e;border-radius:12px;overflow:hidden;background:#090f1d}
    canvas{width:100%;height:100%;display:block;cursor:grab;touch-action:none}
    canvas:active{cursor:grabbing}
    .tip{position:absolute;left:14px;top:13px;max-width:300px;padding:10px 12px;border-radius:9px;background:#0c1830d9;color:#dbe9ff;font-size:12px;pointer-events:none;line-height:1.6}
    .legend{position:absolute;right:14px;top:13px;padding:10px;border-radius:9px;background:#0c1830cc;font-size:12px;line-height:1.8;pointer-events:none}
    .dot{display:inline-block;width:10px;height:10px;border-radius:50%;margin-right:7px}
    .timeline{padding-top:14px}.timeline input{flex:1;min-width:130px;accent-color:#3bcdf7}
    .stats{display:grid;grid-template-columns:repeat(4,minmax(0,1fr));gap:10px;margin-top:12px}
    .stat{background:#121d32;border:1px solid #263a59;padding:11px;border-radius:10px;min-width:0}
    .stat b{font-size:17px;font-variant-numeric:tabular-nums;display:block;overflow:hidden;text-overflow:ellipsis}
    .stat span{color:#93a7c9;font-size:11px}
    .footer{color:#9aaecb;font-size:11px;margin-top:12px;line-height:1.5}
    .stat b{font-size:15px}.stat.wide{grid-column:span 2}
    .panels{display:grid;grid-template-columns:repeat(2,minmax(0,1fr));gap:13px;margin-top:14px}
    .panel{padding:12px;border:1px solid #30405e;border-radius:12px;background:#111a2d;min-width:0}
    .panel header{margin:0 0 10px;gap:6px}.panel h2{font-size:14px;margin:0;color:#e1ebff}
    .chart{width:100%;height:218px;display:block;cursor:crosshair}
    .note{font-size:11px;color:#9dadc9;margin-top:6px;line-height:1.45}
    .phase{display:inline-block;border-radius:7px;background:#1b314b;padding:5px 9px;color:#e0eeff;font-size:11px}
    .coords{display:grid;grid-template-columns:repeat(2,minmax(0,1fr));gap:10px;margin-top:10px}
    .coord{border:1px solid #263958;border-radius:10px;background:#121d32;padding:12px;color:#aabbd5;font-size:11px}
    .coord strong{font-size:12px;color:#e6edff;display:block;margin-bottom:7px}
    .coord code{color:#e5eeff;font-family:ui-monospace,SFMono-Regular,monospace;font-size:12px;white-space:pre-wrap;overflow-wrap:anywhere}
    @media(max-width:770px){.panels{grid-template-columns:1fr}}
    @media(max-width:660px){.scene{height:420px}.stats{grid-template-columns:repeat(2,1fr)}.coords{grid-template-columns:1fr}.legend{font-size:10px}.tip{font-size:10px;max-width:220px}}
    </style></head><body>
    <div id="v2">
    <header><div><h1>VOYAGER 2 · TRAYECTORIA 3D</h1><small>Vista interactiva heliocéntrica o jovicéntrica, sin WebGL</small></div>
    <div class="controls"><label for="modelo">Trayectoria</label>
    <select id="modelo"><option>Horizons</option><option>Modelo 1</option><option selected>Modelo 2</option></select>
    <label for="foco">Cámara</label>
    <select id="foco" aria-label="Centro de cámara">
    <option value="jupiter" selected>Seguir Júpiter</option>
    <option value="voyager">Seguir Voyager 2</option>
    <option value="sol">Seguir el Sol</option>
    <option value="libre">Libre / desplazar</option></select>
    <button id="global">SOI completa</button><button id="cerca">Ver periapsis</button>
    <button id="zoomIn" title="Acercar">+</button><button id="zoomOut" title="Alejar">−</button></div></header>
    <div class="scene"><canvas id="lienzo"></canvas>
    <div class="tip"><strong>Cámara centrada en Júpiter</strong><br>
    Arrastra: girar · <b>Mayús + arrastrar</b> o botón derecho: trasladar<br>
    Rueda: acercar/alejar · Doble clic: volver a Júpiter<br>
    En la vista «Seguir Júpiter», el centro acompaña su posición.<br>
    Flecha amarilla: dirección de velocidad relativa</div>
    <div class="legend"><div><i class="dot" style="background:#d2d4df"></i>Horizons</div>
    <div><i class="dot" style="background:#ff995b"></i>Modelo 1</div>
    <div><i class="dot" style="background:#20dbc3"></i>Modelo 2</div>
    <div><i class="dot" style="background:#ff4765"></i>Voyager 2</div></div></div>
    <div class="timeline"><span id="fase" class="phase">—</span>
    <button id="play">▶ Reproducir</button><button id="back">◀</button>
    <input id="frame" type="range" min="0" max="179" value="0" step="1">
    <button id="next">▶</button><span id="contador" style="min-width:85px;text-align:right">1 / 180</span></div>
    <div class="stats">
    <div class="stat"><span>Fecha (TDB)</span><b id="fecha">—</b></div>
    <div class="stat"><span>Tiempo desde entrada SOI (días)</span><b id="dias">—</b></div>
    <div class="stat"><span>Distancia al Sol (UA)</span><b id="rsol">—</b></div>
    <div class="stat"><span>Distancia a Júpiter (10⁶ km)</span><b id="dist">—</b></div>
    <div class="stat"><span>Rapidez heliocéntrica (km/s)</span><b id="rapH">—</b></div>
    <div class="stat"><span>Rapidez relativa a Júpiter (km/s)</span><b id="rapJ">—</b></div>
    <div class="stat"><span>Energía cinética solar K☉ (km²/s²)</span><b id="Ksun">—</b></div>
    <div class="stat"><span>Energía potencial solar U☉ (km²/s²)</span><b id="Usun">—</b></div>
    <div class="stat"><span>Energía solar ε☉ = K☉ + U☉</span><b id="Esun">—</b></div>
    <div class="stat"><span>Variación Δε☉ desde entrada SOI</span><b id="deltaE">—</b></div>
    <div class="stat"><span>Energía jovicéntrica εJ (km²/s²)</span><b id="Ej">—</b></div>
    <div class="stat"><span>Hasta el radio objetivo 9.58 UA (UA)</span><b id="restante">—</b></div>
    <div class="stat"><span>|h☉| (10⁹ km²/s)</span><b id="hSun">—</b></div>
    <div class="stat"><span>|hJ| (10⁶ km²/s)</span><b id="hJ">—</b></div>
    <div class="stat"><span>Error de posición respecto a Horizons (10⁶ km)</span><b id="errPos">—</b></div>
    <div class="stat"><span>Error de velocidad vectorial frente a Horizons (km/s)</span><b id="errVel">—</b></div>
    </div>
    <div class="coords">
    <div class="coord"><strong id="posTitle">Posición heliocéntrica (UA)</strong><code id="posXYZ">—</code></div>
    <div class="coord"><strong id="velTitle">Velocidad heliocéntrica (km/s)</strong><code id="velXYZ">—</code></div>
    </div>
    <div class="timeline" style="justify-content:space-between;margin-top:12px">
    <span>Intervalo de las gráficas:</span>
    <select id="ventana"><option value="todo">Toda la propagación</option>
    <option value="soi">Dentro de la SOI</option>
    <option value="20">Periapsis ±20 días</option>
    <option value="90" selected>Periapsis ±90 días</option></select>
    <span class="note">Haz clic en las gráficas para cambiar el instante</span>
    </div>
    <div class="panels">
    <section class="panel"><header><h2>Energía específica y conservación</h2>
    <select id="tipoEnergia"><option value="sun">Marco heliocéntrico</option>
    <option value="jov">Marco jovicéntrico</option></select></header>
    <canvas class="chart" id="energyChart"></canvas>
    <div class="note" id="notaEnergia">K, U y ε = K+U, en km²/s² (por unidad de masa).</div></section>
    <section class="panel"><header><h2>Trayectoria y dinámica</h2>
    <select id="tipoGrafica">
    <option value="speed">Rapideces (km/s)</option>
    <option value="jdist">Distancia a Júpiter (10⁶ km)</option>
    <option value="radius">Distancia al Sol (UA)</option>
    <option value="target">Distancia radial restante a 9.58 UA</option>
    <option value="momentumSun">Momento angular solar (10⁹ km²/s)</option>
    <option value="momentumJ">Momento angular jovicéntrico (10⁶ km²/s)</option>
    <option value="error">Error posición vs Horizons (10⁶ km)</option>
    </select></header>
    <canvas class="chart" id="orbitChart"></canvas>
    <div class="note">Líneas y cursor sincronizados con la animación 3D.</div></section>
    </div>
    <div class="footer">K y U son energías <b>específicas</b> (km²/s²): no representan julios sin conocer la masa de la nave.
    Las energías y momentos angulares mostrados se calculan en sus respectivos marcos de referencia;
    ε☉ y εJ representan magnitudes osculatrices de dos cuerpos, no la energía total conservada del sistema completo.
    En el Modelo 1, εJ y hJ son invariantes solamente <b>dentro de la SOI</b>. El radio de 9.58 UA es una referencia geométrica: <b>Saturno no se modela</b>.
    El vector de velocidad y el tamaño de Júpiter/Sol no están a escala.
    Las posiciones mostradas corresponden a las efemérides eclípticas J2000 y los tiempos a TDB.</div>
    </div>
    <script>
    (function(){
    'use strict';
    const data=__PAYLOAD__;
    const colors={'Horizons':'#d2d4df','Modelo 1':'#ff995b','Modelo 2':'#20dbc3'};
    const keys=Object.keys(data.series);
    const el=(id)=>document.getElementById(id);
    const canvas=el('lienzo');
    const ctx=canvas.getContext('2d');
    if(!ctx){el('fecha').textContent='Canvas 2D no disponible';return;}
    const n=data.dates.length;
    el('frame').max=n-1;
    let selected='Modelo 2',idx=data.peri,timer=null,yaw=-0.28,pitch=0.52;
    let W=1000,H=530,span=(data.mode==='jovicentrica'?data.soi*1.35:data.soi*2.8);
    let dragging=false,panDrag=false,px=0,py=0;
    let center=[0,0,0],focus='jupiter',offset=[0,0,0];
    const fullSpan=data.fullSpan;
    function followCenter(){
     if(focus==='jupiter')return data.planetJ[idx];
     if(focus==='voyager')return data.series[selected].pos[idx];
     if(focus==='sol')return data.planetSun[idx];
     return center;
    }
    function updateCenter(){
     if(focus!=='libre')center=followCenter().map((x,k)=>x+offset[k]);
    }
    function setFocus(next,resetOffset=true){
     focus=next;
     if(resetOffset)offset=[0,0,0];
     if(focus!=='libre')updateCenter();
     el('foco').value=focus;
    }
    updateCenter();
    if(data.mode==='jovicentrica')el('ventana').value='soi';
    const perDay=data.periDay;
    const num=(x,d=3)=>Number.isFinite(x)?Number(x).toLocaleString('es-CO',{minimumFractionDigits:d,maximumFractionDigits:d}):'—';
    const axisNum=x=>Math.abs(x)>=10000?x.toExponential(1):Number(x.toFixed(Math.abs(x)<1?3:1)).toString();
    const energyCanvas=el('energyChart'),orbitCanvas=el('orbitChart');
    const ec=energyCanvas.getContext('2d'),oc=orbitCanvas.getContext('2d');
    const MARGIN={left:63,right:13,top:19,bottom:34};
    let chartBox={};
    function getSpan(){
      const kind=el('ventana').value;
      if(kind==='todo')return [data.days[0],data.days[n-1]];
      if(kind==='soi')return [0,data.outDay];
      const d=Number(kind);return [Math.max(data.days[0],perDay-d),Math.min(data.days[n-1],perDay+d)];
    }
    function chartSeries(which){
      const s=data.series[selected];
      if(which==='energy'){
         if(el('tipoEnergia').value==='jov'){
            el('notaEnergia').textContent='Júpiter: KJ, UJ y εJ = KJ + UJ. Conservación de Modelo 1 solo dentro de la SOI.';
            return [{key:'Kj',label:'Cinética KJ',color:'#66c9fd'},
                    {key:'Uj',label:'Potencial UJ',color:'#ffbb75'},
                    {key:'Ej',label:'Mecánica εJ',color:'#86e2b1'}];
         }
         el('notaEnergia').textContent='Sol: K☉, U☉ y ε☉=K☉+U☉ (km²/s²). ε☉ puede cambiar por perturbaciones planetarias.';
         return [{key:'Ksun',label:'Cinética K☉',color:'#66c9fd'},
                 {key:'Usun',label:'Potencial U☉',color:'#ffbb75'},
                 {key:'Esun',label:'Mecánica ε☉',color:'#86e2b1'}];
      }
      switch(el('tipoGrafica').value){
        case 'speed':return [{key:'rapH',label:'Heliocéntrica',color:'#66c9fd'},
                              {key:'rapJ',label:'Relativa a Júpiter',color:'#ffbb75'}];
        case 'jdist':return [{key:'dJ',label:'Distancia a Júpiter',color:'#ffbb75'}];
        case 'radius':return [{key:'rSol',label:'Radio heliocéntrico',color:'#66c9fd'}];
        case 'target':return [{key:'restante',label:'Restante hasta 9.58 UA',color:'#b6a3ff'}];
        case 'momentumSun':return [{key:'hSun',label:'|h☉|',color:'#72dabf'}];
        case 'momentumJ':return [{key:'hJ',label:'|hJ|',color:'#72dabf'}];
        case 'error':return [{key:'errPos',label:'Δr frente a Horizons',color:'#ff9b79'}];
      }
      return [];
    }
    function drawChart(can,context,which){
     if(!context)return;
     let rect=can.getBoundingClientRect(),w=Math.max(240,rect.width),h=Math.max(140,rect.height);
     const dpr=Math.min(2,window.devicePixelRatio||1),pw=Math.round(w*dpr),ph=Math.round(h*dpr);
     if(can.width!==pw || can.height!==ph){can.width=pw;can.height=ph;}
     context.setTransform(dpr,0,0,dpr,0,0);
     context.clearRect(0,0,w,h);
     let x0=MARGIN.left,x1=w-MARGIN.right,y0=MARGIN.top+19,y1=h-MARGIN.bottom;
     let [tmin,tmax]=getSpan();
     const si=data.series[selected];
     const series=chartSeries(which);
     let minY=Infinity,maxY=-Infinity,hasData=false;
     for(const curve of series){
       const values=si[curve.key];if(!values)continue;
       for(let k=0;k<n;k++)if(data.days[k]>=tmin && data.days[k]<=tmax && Number.isFinite(values[k])){
         minY=Math.min(minY,values[k]);maxY=Math.max(maxY,values[k]);hasData=true;
       }
     }
     if(!hasData){context.font='12px system-ui';context.fillStyle='#a5b8d5';
        context.fillText('Sin datos en este intervalo',18,h/2);return;}
     if(Math.abs(maxY-minY)<1e-11){const pad=Math.max(Math.abs(maxY)*.02,.01);minY-=pad;maxY+=pad;}
     const pad=(maxY-minY)*.08;minY-=pad;maxY+=pad;
     if(minY<0 && series.every(c=>['rapH','rapJ','dJ','rSol','hSun','hJ','errPos'].includes(c.key)))minY=0;
     const XP=t=>x0+(t-tmin)/(tmax-tmin)*(x1-x0);
     const YP=v=>y1-(v-minY)/(maxY-minY)*(y1-y0);
     context.font='10px system-ui';context.textBaseline='middle';
     for(let j=0;j<=4;j++){
       const v=minY+(maxY-minY)*j/4;const y=YP(v);
       context.beginPath();context.moveTo(x0,y);context.lineTo(x1,y);
       context.strokeStyle='#2a3954';context.lineWidth=1;context.stroke();
       context.fillStyle='#a3b2d0';context.textAlign='right';context.fillText(axisNum(v),x0-7,y);
     }
     for(let j=0;j<=4;j++){
       const tt=tmin+(tmax-tmin)*j/4;const x=XP(tt);
       context.beginPath();context.moveTo(x,y0);context.lineTo(x,y1);
       context.strokeStyle='#1b2a46';context.stroke();
       context.fillStyle='#a3b2d0';context.textAlign='center';context.fillText(axisNum(tt),x,y1+15);
     }
     context.fillStyle='#90a6c5';context.textAlign='center';context.fillText('Días desde entrada SOI', (x0+x1)/2,h-5);
     for(const curve of series){
       const values=si[curve.key];if(!values)continue;
       context.beginPath();let active=false;
       for(let k=0;k<n;k++){
         if(data.days[k]<tmin || data.days[k]>tmax || !Number.isFinite(values[k])){active=false;continue;}
         const x=XP(data.days[k]),y=YP(values[k]);
         if(!active){context.moveTo(x,y);active=true;}else context.lineTo(x,y);
       }
       context.strokeStyle=curve.color;context.lineWidth=2;context.stroke();
     }
     // Marca temporal seleccionada y referencia a la máxima aproximación.
     for(const ref of [{time:perDay,color:'#c9b8e7',dash:[2,4]},
                       {time:data.days[idx],color:'#f8f8f8',dash:[5,4]}]){
        if(ref.time<tmin||ref.time>tmax)continue;
        const x=XP(ref.time);context.setLineDash(ref.dash);
        context.beginPath();context.moveTo(x,y0);context.lineTo(x,y1);
        context.strokeStyle=ref.color;context.lineWidth=1.2;context.stroke();context.setLineDash([]);
     }
     let lx=x0;
     context.font='11px system-ui';context.textAlign='left';
     for(const c of series){
       context.fillStyle=c.color;context.fillRect(lx,MARGIN.top-10,9,2);
       context.fillStyle='#d9e8ff';context.fillText(c.label,lx+13,MARGIN.top-9);
       lx+=Math.min(175,36+c.label.length*6.4);
     }
     chartBox[which]={canvas:can,tmin,tmax,x0,x1};
    }
    function drawCharts(){drawChart(energyCanvas,ec,'energy');drawChart(orbitCanvas,oc,'orbit');}
    function jumpChart(e,which){const c=chartBox[which];if(!c)return;
      const rect=c.canvas.getBoundingClientRect(),x=e.clientX-rect.left;
      const tt=c.tmin+(Math.min(c.x1,Math.max(c.x0,x))-c.x0)/(c.x1-c.x0)*(c.tmax-c.tmin);
      let best=0,dist=Infinity;
      for(let k=0;k<n;k++){let d=Math.abs(data.days[k]-tt);if(d<dist){dist=d;best=k;}}
      pause();idx=best;refresh();
    }
    energyCanvas.addEventListener('click',e=>jumpChart(e,'energy'));
    orbitCanvas.addEventListener('click',e=>jumpChart(e,'orbit'));
    el('tipoEnergia').addEventListener('change',drawCharts);
    el('tipoGrafica').addEventListener('change',drawCharts);
    el('ventana').addEventListener('change',drawCharts);

    if(data.mode==='heliocentrica'){
     document.querySelector('#v2 h1').textContent='VOYAGER 2 · SOL–JÚPITER–SONDA';
     document.querySelector('#v2 header small').textContent='Trayectoria heliocéntrica 3D · ejes en UA · sin WebGL';
     el('cerca').textContent='Acercar a Júpiter';
    }else{
     document.querySelector('#v2 h1').textContent='VOYAGER 2 · JÚPITER';
     document.querySelector('#v2 header small').textContent='Encuentro jovicéntrico 3D · ejes en millones de km · sin WebGL';
    }
    const stars=Array.from({length:105},(_,i)=>[ ((i*97.3)%997)/997, ((i*61.7)%571)/571, 0.25+((i*17)%13)/25 ]);
    function rotate(p){
      const c=Math.cos(yaw),s=Math.sin(yaw),cp=Math.cos(pitch),sp=Math.sin(pitch);
      const x=c*p[0]-s*p[1], y=s*p[0]+c*p[1],z=p[2];
      return [x,cp*y-sp*z,sp*y+cp*z];
    }
    function project(p){
      const r=rotate(p.map((v,i)=>v-center[i]));
      const focal=span*5.8;
      const perspective=focal/Math.max(span,focal-r[2]);
      const factor=Math.min(W*0.44,H*0.44)/span;
      return [W/2+r[0]*factor*perspective,H/2-r[1]*factor*perspective];
    }
    function path3D(points,col,width,opacity,dash){
      if(points.length<2)return;
      ctx.beginPath();
      for(let i=0;i<points.length;i++){
        const p=project(points[i]);
        if(i===0)ctx.moveTo(p[0],p[1]);else ctx.lineTo(p[0],p[1]);
      }
      ctx.strokeStyle=col;ctx.globalAlpha=opacity;ctx.lineWidth=width;
      ctx.setLineDash(dash||[]);ctx.stroke();ctx.setLineDash([]);ctx.globalAlpha=1;
    }
    function marker(p,r,col){
      const q=project(p);const glow=ctx.createRadialGradient(q[0]-r*.3,q[1]-r*.3,1,q[0],q[1],r);
      glow.addColorStop(0,'#fff4df');glow.addColorStop(.28,col);glow.addColorStop(1,col+'00');
      ctx.beginPath();ctx.arc(q[0],q[1],r,0,Math.PI*2);ctx.fillStyle=glow;ctx.fill();
      ctx.beginPath();ctx.arc(q[0],q[1],Math.max(3,r*.35),0,Math.PI*2);ctx.fillStyle=col;ctx.fill();
    }
    function arrow(p,dir){
      const mag=Math.hypot(...dir);if(mag<1e-12)return;
      const a=project(p),b=project(p.map((v,i)=>v+dir[i]/mag*span*.22));
      let dx=b[0]-a[0],dy=b[1]-a[1],dn=Math.hypot(dx,dy);
      if(dn<1)return;dx/=dn;dy/=dn;
      const end=[a[0]+dx*76,a[1]+dy*76];
      ctx.beginPath();ctx.moveTo(...a);ctx.lineTo(...end);ctx.lineWidth=3.4;
      ctx.strokeStyle='#ffeb5e';ctx.stroke();
      ctx.beginPath();ctx.moveTo(...end);
      ctx.lineTo(end[0]-dx*13+dy*6,end[1]-dy*13-dx*6);
      ctx.lineTo(end[0]-dx*13-dy*6,end[1]-dy*13+dx*6);
      ctx.closePath();ctx.fillStyle='#ffeb5e';ctx.fill();
    }
    function axes(){
      // Ejes locales sobre el centro de cámara: permiten ver en qué espacio estamos.
      for(let j=0;j<3;j++){
        const u=[0,0,0];u[j]=span*.68;
        path3D([u.map((v,i)=>center[i]-v),u.map((v,i)=>center[i]+v)],
            ['#e06272','#59b892','#608fea'][j],1.4,.5);
        const pp=project(u.map((v,i)=>center[i]+v));ctx.font='12px system-ui';ctx.fillStyle=['#e88e9b','#79d8ad','#89b3ff'][j];
        ctx.fillText(['X','Y','Z'][j],pp[0]+7,pp[1]-5);
      }
    }
    function soiRings(){
      const rad=data.soi;
      if(span<rad*.5)return;
      const origin=data.planetJ[idx];
      const segments=72;
      for(let pl=0;pl<3;pl++){
        const ring=[];
        for(let k=0;k<=segments;k++){
          const phi=k*2*Math.PI/segments;
          const a=rad*Math.cos(phi),b=rad*Math.sin(phi);
          const pt=pl===0?[a,b,0]:pl===1?[a,0,b]:[0,a,b];
          ring.push(pt.map((v,i)=>v+origin[i]));
        }
        path3D(ring,'#64738a',1,.23,[4,6]);
      }
    }
    function draw(){
      updateCenter();
      ctx.clearRect(0,0,W,H);
      const bg=ctx.createLinearGradient(0,0,W,H);bg.addColorStop(0,'#0a1328');bg.addColorStop(1,'#050916');
      ctx.fillStyle=bg;ctx.fillRect(0,0,W,H);
      for(let i=0;i<stars.length;i++){
        const s=stars[i];ctx.globalAlpha=s[2];ctx.fillStyle='#d7e6ff';
        ctx.fillRect(s[0]*W,s[1]*H,i%8===0?2:1,i%8===0?2:1);
      }ctx.globalAlpha=1;
      soiRings();axes();
      if(data.mode==='heliocentrica'){
        const radiusGuide=[];
        for(let k=0;k<=120;k++){
          const a=k*Math.PI*2/120;radiusGuide.push([9.58*Math.cos(a),9.58*Math.sin(a),0]);
        }
        path3D(radiusGuide,'#968ac6',1,.28,[4,7]);
        path3D(data.planetJ,'#eead70',1,.3,[3,5]);
        marker([0,0,0],12,'#ffd46a');
        const sun=project([0,0,0]);ctx.fillStyle='#ffe5a7';ctx.font='12px system-ui';
        ctx.fillText('Sol',sun[0]+16,sun[1]+14);
        marker(data.planetJ[idx],8,'#d98b55');
        let jt=project(data.planetJ[idx]);ctx.fillStyle='#e6ad7e';
        ctx.fillText('Júpiter',jt[0]+12,jt[1]+12);
        ctx.fillStyle='#9399bd';ctx.font='11px system-ui';
        ctx.fillText('Radio geométrico de referencia: 9.58 UA (sin Saturno)',14,H-15);
      }
      for(const key of keys){
        const points=data.series[key].pos;
        path3D(points,colors[key],key===selected?2.6:1.4,key===selected?1:.38,key===selected?[]:[5,6]);
      }
      const s=data.series[selected];
      path3D(s.pos.slice(0,idx+1),'#00f6c4',4.4,.97);
      if(data.mode==='jovicentrica'){
        marker([0,0,0],12,'#e5a05e');
        const q=project([0,0,0]);ctx.fillStyle='#ffc88e';ctx.font='12px system-ui';
        ctx.fillText('Júpiter',q[0]+17,q[1]+18);
      }
      arrow(s.pos[idx],s.vel[idx]);marker(s.pos[idx],8,'#ff4765');
    }
    function refresh(){
      updateCenter();
      el('frame').value=idx;
      el('contador').textContent=(idx+1)+' / '+n;
      const s=data.series[selected],day=data.days[idx];
      let phase='Antes del encuentro';
      if(day>=0 && day<=data.outDay)phase='Dentro de la SOI de Júpiter';
      else if(day>data.outDay)phase='Después del encuentro';
      if(Math.abs(day-perDay)<0.8)phase+=' · periapsis';
      el('fase').textContent=phase+' · centro: '+({jupiter:'Júpiter',voyager:'Voyager 2',sol:'Sol',libre:'libre'}[focus]);
      el('fecha').textContent=data.dates[idx];
      el('dias').textContent=num(day,2);
      el('dist').textContent=num(s.dJ[idx],3);
      el('rsol').textContent=num(s.rSol[idx],5);
      el('rapH').textContent=num(s.rapH[idx],4);
      el('rapJ').textContent=num(s.rapJ[idx],4);
      el('Ksun').textContent=num(s.Ksun[idx],4);
      el('Usun').textContent=num(s.Usun[idx],4);
      el('Esun').textContent=num(s.Esun[idx],4);
      const ie=data.days.findIndex(d=>d>=0),base=ie>=0?s.Esun[ie]:s.Esun[0];
      el('deltaE').textContent=num(s.Esun[idx]-base,4);
      el('Ej').textContent=num(s.Ej[idx],5);
      el('restante').textContent=num(s.restante[idx],5);
      el('hSun').textContent=num(s.hSun[idx],4);
      el('hJ').textContent=num(s.hJ[idx],4);
      el('errPos').textContent=num(s.errPos[idx],5);
      el('errVel').textContent=num(s.errVel[idx],5);
      const position=data.mode==='heliocentrica'?s.rHelio[idx]:s.rJov[idx];
      const velocity=data.mode==='heliocentrica'?s.vHelio[idx]:s.vJov[idx];
      el('posTitle').textContent=data.mode==='heliocentrica'?'Posición heliocéntrica (UA)':'Posición jovicéntrica (10⁶ km)';
      el('velTitle').textContent=data.mode==='heliocentrica'?'Velocidad heliocéntrica (km/s)':'Velocidad respecto a Júpiter (km/s)';
      el('posXYZ').textContent=`X = ${num(position[0],5)}   Y = ${num(position[1],5)}   Z = ${num(position[2],5)}`;
      el('velXYZ').textContent=`VX = ${num(velocity[0],5)}   VY = ${num(velocity[1],5)}   VZ = ${num(velocity[2],5)}`;
      draw();drawCharts();
    }
    function pause(){if(timer!==null){clearInterval(timer);timer=null;}el('play').textContent='▶ Reproducir';}
    function play(){
      if(timer!==null){pause();return;}
      if(idx===n-1)idx=0;
      timer=setInterval(()=>{idx++;if(idx>=n){idx=n-1;pause();}refresh();},data.frameMs);
      el('play').textContent='⏸ Pausa';
    }
    el('play').onclick=play;
    el('back').onclick=()=>{pause();idx=Math.max(0,idx-1);refresh();};
    el('next').onclick=()=>{pause();idx=Math.min(n-1,idx+1);refresh();};
    el('frame').oninput=(e)=>{pause();idx=Number(e.target.value);refresh();};
    el('modelo').onchange=(e)=>{selected=e.target.value;refresh();};
    el('foco').onchange=(e)=>{
      setFocus(e.target.value);
      if(focus==='sol' && data.mode==='jovicentrica')span=Math.max(span,800);
      if(focus==='jupiter')span=Math.min(span,data.soi*2.1);
      refresh();
    };
    el('global').onclick=()=>{
      setFocus('jupiter');idx=data.peri;
      span=data.soi*1.35;refresh();
    };
    el('cerca').onclick=()=>{
      setFocus('jupiter');idx=data.peri;
      span=data.mode==='heliocentrica'?Math.max(.012,data.periDist*8):Math.max(1.4,data.periDist*7);
      refresh();
    };
    el('zoomIn').onclick=()=>{span=Math.max(.0004,span*.7);draw();};
    el('zoomOut').onclick=()=>{span=Math.min(fullSpan*10,span*1.4);draw();};
    // Arrastre normal = rotación. Mayúsculas/botón derecho = desplazamiento real
    // de la cámara en unidades del gráfico (no solo mover el ratón sobre la escena).
    function pan(dx,dy){
     const factor=Math.min(W*.44,H*.44)/span;
     const a=-dx/factor, b=dy/(factor*Math.max(.19,Math.cos(pitch)));
     const c=Math.cos(yaw),s=Math.sin(yaw);
     center[0]+=c*a+s*b;
     center[1]+=-s*a+c*b;
     setFocus('libre',false);
    }
    canvas.addEventListener('contextmenu',(e)=>e.preventDefault());
    canvas.onpointerdown=(e)=>{
     dragging=true;panDrag=(e.shiftKey||e.button===2);px=e.clientX;py=e.clientY;
     canvas.setPointerCapture(e.pointerId);
    };
    canvas.onpointerup=()=>{dragging=false;panDrag=false;};
    canvas.onpointercancel=()=>{dragging=false;panDrag=false;};
    canvas.onpointermove=(e)=>{
     if(!dragging)return;
     const dx=e.clientX-px,dy=e.clientY-py;
     if(panDrag||e.shiftKey){pan(dx,dy);}else{
       yaw+=dx*.008;pitch=Math.max(-1.52,Math.min(1.52,pitch+dy*.008));
     }
     px=e.clientX;py=e.clientY;draw();
    };
    canvas.onwheel=(e)=>{
     e.preventDefault();span=Math.max(.0004,Math.min(fullSpan*10,span*(e.deltaY>0?1.15:.87)));draw();
    };
    canvas.ondblclick=()=>{
     yaw=-.28;pitch=.52;idx=data.peri;
     setFocus('jupiter');span=data.soi*1.35;refresh();
    };
    function resize(){
     const box=canvas.getBoundingClientRect();W=box.width;H=box.height;
     const dpr=Math.min(2,window.devicePixelRatio||1);
     canvas.width=Math.round(W*dpr);canvas.height=Math.round(H*dpr);
     ctx.setTransform(dpr,0,0,dpr,0,0);draw();
    }
    window.addEventListener('resize',resize);
    resize();refresh();
    })();
    </script></body></html>'''.replace("__PAYLOAD__", _payload_json)

    _destino = _result_dir / ARCHIVO_HTML
    _destino.write_text(_documento, encoding="utf-8")

    # Iframe srcdoc evita conflicto entre scripts y elementos de otras celdas.
    if mostrar:
        display(HTML(
            f'<iframe title="Voyager 2: cámara centrada en Júpiter sin WebGL" '
            f'style="width:100%;height:1780px;border:0;border-radius:12px" '
            f'srcdoc="{html.escape(_documento, quote=True)}"></iframe>'
        ))
    print("HTML interactivo sin WebGL generado:", _destino)
    print("Controles: seguir Júpiter/Voyager/Sol; Shift+arrastrar para trasladar; girar, zoom y reproducir.")

    return _destino, _tele_csv


## 5. HTML principal: sobrevuelo **centrado en Júpiter**

El visor queda embebido aquí y se guarda en un HTML autónomo que podrás abrir desde la carpeta de resultados o el ZIP descargado. Al pasar a modo libre podrás desplazarte sin quedar limitado a un centro fijo.

In [7]:
html_jupiter, tabla_jupiter = generar_panel3d('jovicentrica', mostrar=True)
print('HTML jovicéntrico:', html_jupiter)
print('Telemetría jovicéntrica:', tabla_jupiter)

Telemetría exportada: /content/resultados_voyager2_sol_jupiter/telemetria_dashboard_jovicentrica.csv


/usr/local/lib/python3.13/dist-packages/IPython/core/display.py:724: UserWarning:

Consider using IPython.display.IFrame instead



HTML interactivo sin WebGL generado: /content/resultados_voyager2_sol_jupiter/13_sobrevuelo_jupiter_centrado.html
Controles: seguir Júpiter/Voyager/Sol; Shift+arrastrar para trasladar; girar, zoom y reproducir.
HTML jovicéntrico: /content/resultados_voyager2_sol_jupiter/13_sobrevuelo_jupiter_centrado.html
Telemetría jovicéntrica: /content/resultados_voyager2_sol_jupiter/telemetria_dashboard_jovicentrica.csv


## 6. Segundo HTML: trayectoria heliocéntrica y alcance de 9.58 UA

Esta visualización muestra el Sol, Júpiter y Voyager 2. La circunferencia de 9.58 UA es solo una **referencia radial**; **no aparece Saturno como cuerpo ni se calculan sus perturbaciones**. La cámara comienza siguiendo a Júpiter y ofrece controles libres.

In [8]:
html_helio, tabla_helio = generar_panel3d('heliocentrica', mostrar=False)
print('HTML heliocéntrico:', html_helio)
print('Telemetría heliocéntrica:', tabla_helio)
print('Puedes abrir los HTML desde la carpeta de resultados descargada en la última celda.')

Telemetría exportada: /content/resultados_voyager2_sol_jupiter/telemetria_dashboard_heliocentrica.csv
HTML interactivo sin WebGL generado: /content/resultados_voyager2_sol_jupiter/12_trayectoria_heliocentrica_sin_saturno.html
Controles: seguir Júpiter/Voyager/Sol; Shift+arrastrar para trasladar; girar, zoom y reproducir.
HTML heliocéntrico: /content/resultados_voyager2_sol_jupiter/12_trayectoria_heliocentrica_sin_saturno.html
Telemetría heliocéntrica: /content/resultados_voyager2_sol_jupiter/telemetria_dashboard_heliocentrica.csv
Puedes abrir los HTML desde la carpeta de resultados descargada en la última celda.


## 7. Marco teórico, metodología, cálculos y conclusiones en LaTeX

La siguiente celda guarda el archivo fuente `.tex` **completo** (no solo una plantilla). Incluye demostraciones de energía y momento angular, dispersión hiperbólica, SOI Laplace y Hill, transformación de coordenadas, términos directos e indirectos, análisis de error, método numérico, ejemplos con sustitución de números y conclusiones. Copia también las figuras a `figuras/` para que el `.tex` pueda compilar en Overleaf. Las figuras se vinculan por ruta relativa; no necesitas otros archivos científicos para compilar el texto.

In [9]:
# 7. Escribir informe completo LaTeX y organizar sus figuras
from pathlib import Path
import shutil
SALIDA = Path('/content/resultados_voyager2_sol_jupiter')
RUTA_TEX = SALIDA / 'Informe_Reto_III3_Sol_Jupiter_Voyager2.tex'
TEX_FUENTE = r'''\documentclass[11pt,a4paper]{article}
\usepackage[spanish,es-nodecimaldot]{babel}
\usepackage[utf8]{inputenc}
\usepackage[T1]{fontenc}
\usepackage{lmodern}
\usepackage{microtype}
\usepackage{amsmath,amssymb,amsthm,bm}
\usepackage{geometry}
\usepackage{booktabs,tabularx,array,longtable}
\usepackage{graphicx,float,subcaption}
\usepackage{xcolor}
\usepackage{hyperref}
\usepackage{listings}
\usepackage{enumitem}
\usepackage{fancyhdr}
\geometry{margin=2.35cm,headheight=15pt}
\hypersetup{colorlinks=true,linkcolor=blue!50!black,urlcolor=blue!50!black,citecolor=blue!50!black}
\pagestyle{fancy}
\fancyhf{}
\fancyhead[L]{Reto III.3: asistencia gravitacional}
\fancyhead[R]{Sol--Júpiter--Voyager 2}
\fancyfoot[C]{\thepage}
\setlength{\parskip}{.45em}
\setlength{\parindent}{0pt}
\setlist[itemize]{topsep=3pt,itemsep=2pt}
\definecolor{codebg}{RGB}{244,247,250}
\definecolor{codeblue}{RGB}{20,65,112}
\lstdefinestyle{python}{language=Python,basicstyle=\ttfamily\footnotesize,
  keywordstyle=\color{codeblue}\bfseries,commentstyle=\color{teal!60!black},
  stringstyle=\color{orange!65!black},breaklines=true,
  backgroundcolor=\color{codebg},frame=single,showstringspaces=false,
  numbers=left,numberstyle=\tiny\color{gray},xleftmargin=1.2em,
  literate={á}{{\'a}}1 {é}{{\'e}}1 {í}{{\'i}}1 {ó}{{\'o}}1 {ú}{{\'u}}1 {ñ}{{\~n}}1 {ü}{{\"u}}1}
\lstset{style=python}
\newcommand{\vect}[1]{\bm{#1}}
\newcommand{\dif}{\mathop{}\!\mathrm{d}}
\newcommand{\SOI}{R_{\mathrm{SOI}}}
\newcommand{\muS}{\mu_\odot}
\newcommand{\muJ}{\mu_{J}}
\newcommand{\rJ}{\vect r_{J}}
\newcommand{\vJ}{\vect v_{J}}
\newcommand{\rV}{\vect r}
\newcommand{\vV}{\vect v}
\newcommand{\fign}[3]{\begin{figure}[htbp]\centering\IfFileExists{figuras/#1}{\includegraphics[width=#2\textwidth]{figuras/#1}}{\fbox{\parbox{.85\textwidth}{Archivo opcional: \texttt{figuras/#1}}}}\caption{#3}\end{figure}}
\begin{document}
\begin{titlepage}
\centering
\vspace*{2cm}
{\Large Mecánica clásica y física computacional\par}
\vspace{1.2cm}
{\Huge\bfseries Reto III.3\par}
\vspace{.5cm}
{\LARGE Diseño y validación de una asistencia gravitacional\par}
\vspace{.65cm}
{\Large Caso histórico: Voyager 2 en Júpiter (1979)\par}
\vspace{1.1cm}
\rule{.8\textwidth}{.6pt}\par
\vspace{.5cm}
{\large\bfseries Sistema dinámico: Sol, Júpiter y sonda\par}
{\normalsize Dos aproximaciones: cónicas empalmadas e integración gravitacional\par}
\vspace{1.4cm}
\begin{minipage}{.83\textwidth}
\textbf{Alcance:} se prescriben las efemérides heliocéntricas de Júpiter y se considera la sonda de masa despreciable. El radio $9.58\,\mathrm{UA}$ sirve solo como referencia geométrica de salida hacia la región de Saturno; \emph{no se incluyen efemérides ni aceleración gravitatoria de Saturno}.
\end{minipage}
\vfill
{\large Documento metodológico y resultados reproducibles\par}
{\large Noviembre de 2026\par}
\end{titlepage}
\tableofcontents
\newpage

\section{Resumen, problema y objetivos}
El propósito es reconstruir el sobrevuelo de la sonda Voyager 2 por Júpiter mediante las efemérides de JPL Horizons, modelar la desviación gravitatoria y demostrar cómo cambia su trayectoria en el sistema heliocéntrico. Se comparan dos descripciones que comparten exactamente la misma condición inicial de entrada: (i) una integración de dos cuerpos alrededor de Júpiter limitada por una esfera de influencia, unida a arcos heliocéntricos keplerianos, y (ii) una propagación restringida con la acción simultánea del Sol y Júpiter.

\textbf{Objetivo físico:} determinar qué energía y qué momento angular se conservan, en qué referencia y bajo qué hipótesis. \textbf{Objetivo numérico:} cuantificar el error de la aproximación de cónicas empalmadas y su dependencia del radio de cambio de modelo. \textbf{Objetivo geométrico:} establecer si, después del sobrevuelo, la sonda cruza el radio heliocéntrico $R_{\mathrm{obj}}=9.58\,\mathrm{UA}$. Esto último es una condición necesaria de alcance radial, \emph{no una predicción de encuentro con Saturno}.

\section{Marco teórico}
\subsection{Sistema de referencia, notación y unidades}
Trabajamos en coordenadas heliocéntricas eclípticas J2000 del archivo Horizons, con posiciones en $\mathrm{km}$, velocidades en $\mathrm{km\,s^{-1}}$, tiempos en segundos y época astronómica TDB. Denotamos con $\rV(t)$ y $\vV(t)$ el estado de Voyager y con $\rJ(t)$ y $\vJ(t)$ el de Júpiter, ambos respecto del Sol. La masa de la sonda es despreciable frente a las masas de los primarios.

Definimos el estado relativo jovicéntrico por
\begin{equation}\label{eq:rel}
\vect\rho=\rV-\rJ,\qquad \vect w=\vV-\vJ,
\qquad \rho=\lVert\vect\rho\rVert.
\end{equation}
La ecuación~\eqref{eq:rel} es una traslación galileana instantánea de posiciones y velocidades. El sistema centrado en Júpiter, al acelerarse con su movimiento orbital, \emph{no es inercial} durante un intervalo largo. Por tanto no debe aplicarse directamente la ecuación jovicéntrica pura cuando se quieren incluir todas las fuerzas.

En el código se emplean:
\begin{align*}
\muS &= 1.32712440018\times 10^{11}\ \mathrm{km^3\,s^{-2}},\\
\muJ &= 1.26686534\times 10^{8}\ \mathrm{km^3\,s^{-2}},\\
1\ \mathrm{UA}&=149\,597\,870.7\ \mathrm{km},\qquad 1\ \mathrm{día}=86\,400\ \mathrm{s},\\
a_J&=778.57\times10^6\ \mathrm{km},\\
M_\odot &= 1.98847\times10^{30}\ \mathrm{kg},\qquad M_J=1.89813\times10^{27}\ \mathrm{kg}.
\end{align*}
Se utilizan valores constantes para los parámetros gravitacionales y un semieje mayor representativo de Júpiter. Las efemérides no se sustituyen por órbitas circulares al calcular la trayectoria nominal.

\subsection{Problema de dos cuerpos en un campo central}
Para la posición relativa a un cuerpo de parámetro gravitacional $\mu=GM$, el modelo kepleriano satisface
\begin{equation}\label{eq:kepler}
\ddot{\vect r}=-\frac{\mu}{r^3}\vect r,\qquad
\vect F=-\frac{GMm}{r^3}\vect r.
\end{equation}
La fuerza es central: siempre paralela a $\vect r$. La energía específica (energía por unidad de masa de la sonda) es
\begin{equation}\label{eq:energy}
\varepsilon=\frac{\dot{\vect r}\cdot\dot{\vect r}}2-\frac{\mu}{r}.
\end{equation}
Derivando respecto al tiempo,
\begin{align}
\frac{\dif\varepsilon}{\dif t}
 &=\dot{\vect r}\cdot\ddot{\vect r}+\mu\frac{\vect r\cdot\dot{\vect r}}{r^3}\\
 &=\dot{\vect r}\cdot\left(-\frac\mu{r^3}\vect r\right)
 +\mu\frac{\vect r\cdot\dot{\vect r}}{r^3}=0.
\end{align}
En consecuencia, $\varepsilon$ es constante en un problema de dos cuerpos autónomo. El momento angular específico es el \emph{vector}
\begin{equation}\label{eq:h}
\vect h=\vect r\times\dot{\vect r}.
\end{equation}
Su derivada es
\begin{equation}
\dot{\vect h}=\dot{\vect r}\times\dot{\vect r}+
\vect r\times\ddot{\vect r}=\vect 0,
\end{equation}
porque $\ddot{\vect r}$ es paralelo a $\vect r$. La conservación \emph{vectorial} fija el plano orbital, no solamente el módulo $h=\lVert\vect h\rVert$.

\subsection{Energía, excentricidad y elementos osculatrices}
El vector de excentricidad es
\begin{equation}
\vect e=\frac{\vect v\times\vect h}{\mu}-\frac{\vect r}{r},
\qquad e=\lVert\vect e\rVert.
\end{equation}
La integración del problema central conduce a la ecuación polar de una cónica
\begin{equation}\label{eq:conic}
r(\nu)=\frac{p}{1+e\cos\nu},\qquad p=\frac{h^2}{\mu}.
\end{equation}
La energía determina el semieje mayor
\begin{equation}
\varepsilon=-\frac{\mu}{2a},\qquad a=-\frac{\mu}{2\varepsilon}.
\end{equation}
Para $\varepsilon<0$ la órbita es elíptica ($a>0$), con afelio $r_a=a(1+e)$. Si $\varepsilon>0$, la cónica es hiperbólica ($a<0$, $e>1$) y \emph{no hay afelio finito}. La inclinación, en eclíptica J2000, se calcula mediante
\begin{equation}
i=\arccos\left(\frac{h_z}{\lVert\vect h\rVert}\right).
\end{equation}
Los elementos calculados durante el encuentro se llaman \emph{osculatrices}: representan la cónica instantánea asociada al estado, no una trayectoria exacta si hay perturbaciones.

\subsection{Dispersión hiperbólica por gravitación}
En el sobrevuelo jovicéntrico ideal, la energía relativa es positiva y puede definirse la rapidez asintótica por
\begin{equation}
\varepsilon_J=\frac12w^2-\frac{\muJ}{\rho}>0,
\qquad v_\infty=\sqrt{2\varepsilon_J}.
\end{equation}
Del momento angular $h_J=\rho\,w_\perp$, muy lejos de Júpiter se obtiene el parámetro de impacto
\begin{equation}\label{eq:b}
b=\frac{h_J}{v_\infty}.
\end{equation}
A partir de la energía y del momento angular,
\begin{equation}\label{eq:ecc}
e=\sqrt{1+\frac{2\varepsilon_J h_J^2}{\muJ^2}},\qquad
r_p=\frac{h_J^2}{\muJ(1+e)}.
\end{equation}
Las asíntotas de la hipérbola satisfacen $\cos\nu_\infty=-1/e$. El cambio de dirección entre las velocidades asintóticas es
\begin{equation}\label{eq:gamma}
\gamma=2\arcsin\left(\frac1e\right)=2\arctan\left(\frac{\muJ}{bv_\infty^2}\right).
\end{equation}
Según Berg~\cite{berg2023}, ecuaciones (26)--(27), se define el parámetro adimensional
\begin{equation}\label{eq:D}
D=\frac{\muJ}{b v_\infty^2},\qquad \gamma=2\arctan D.
\end{equation}
Así, $D\ll1$ representa una dispersión débil y $D\gg1$ una gran desviación. Es importante separar $\gamma$, \emph{ángulo entre asíntotas}, del giro medido entre velocidades sobre la \emph{frontera finita} de la SOI: estas cantidades no tienen por qué coincidir.

\subsection{Por qué una sonda gana energía heliocéntrica}
Sean $\vect w_-$ y $\vect w_+$ las velocidades relativas a Júpiter antes y después del sobrevuelo, tomadas suficientemente lejos del planeta, y sea $\vJ$ su velocidad heliocéntrica aproximadamente constante durante la maniobra corta. En esa aproximación,
\begin{equation}
\vect v_-=\vJ+\vect w_-,\qquad
\vect v_+=\vJ+\vect w_+,\qquad |\vect w_-|=|\vect w_+|=v_\infty.
\end{equation}
Por tanto, la diferencia de energía cinética heliocéntrica específica es
\begin{align}\label{eq:transfer}
\Delta K_\odot
&=\frac12|\vect v_+|^2-\frac12|\vect v_-|^2\\
&=\vJ\cdot(\vect w_+-\vect w_-)
+\frac12\bigl(|\vect w_+|^2-|\vect w_-|^2\bigr)\\
&=\boxed{\vJ\cdot(\vect w_+-\vect w_-)}.
\end{align}
La sonda puede ganar energía heliocéntrica porque Júpiter está en movimiento; en un tratamiento de masas finitas se intercambian cantidades muy pequeñas de energía y momento con el planeta. Esto \textbf{no viola} la conservación de la energía del sistema físico completo.

Berg~\cite{berg2023} introduce la geometría plana con la rapidez planetaria $u_p$, la rapidez asintótica relativa $u_i$, el ángulo jovicéntrico de entrada $\beta$ y el giro $\gamma$. La ecuación (32) del artículo equivale a
\begin{multline}\label{eq:berg32}
\Delta u=\sqrt{u_p^2+u_i^2-2u_pu_i\cos(\beta+\gamma)}\\
-\sqrt{u_p^2+u_i^2-2u_pu_i\cos\beta}.
\end{multline}
Usando
\begin{equation}
\cos\gamma=\frac{1-D^2}{1+D^2},\qquad
\sin\gamma=\frac{2D}{1+D^2},
\end{equation}
se transforma la ecuación~\eqref{eq:berg32} en la ecuación (37) de Berg,
\begin{multline}\label{eq:berg37}
\Delta u=\Biggl[u_p^2+u_i^2-2u_pu_i\left(
\frac{1-D^2}{1+D^2}\cos\beta-\frac{2D}{1+D^2}\sin\beta\right)\Biggr]^{1/2}\\
-\sqrt{u_p^2+u_i^2-2u_pu_i\cos\beta}.
\end{multline}
Para relacionar la entrada heliocéntrica con la jovicéntrica (ecuaciones (39)--(42)), el plano se orienta con eje tangencial al movimiento de Júpiter y eje radial:
\begin{align}
u'_{m,x}&=u'_m\sin\alpha,& u_{m,x}&=u'_{m,x}-u_p,\\
u_i&=\sqrt{u_{m,x}^2+(u'_m\cos\alpha)^2},&
\cos\beta&=\frac{|u_{m,x}|}{u_i}.
\end{align}
\textbf{Advertencia de validación:} estas expresiones suponían una orientación concreta del giro, un plano de dispersión y velocidad planetaria aproximadamente tangencial. En el código se comprueban además las velocidades asintóticas \emph{vectoriales 3D} y la ganancia entre superficies finitas de la SOI. No se debe obligar a que estas cuatro evaluaciones sean exactamente iguales.

\subsection{Esfera de influencia: Laplace y Hill}
La aproximación de cónicas empalmadas necesita una frontera convencional. Se adopta el radio de Laplace,
\begin{equation}\label{eq:soi}
\SOI=a_J\left(\frac{M_J}{M_\odot}\right)^{2/5},
\end{equation}
obtenido en el tratamiento clásico de la dominancia de perturbaciones relativas. Como escala alternativa se calcula el radio de Hill para una órbita aproximadamente circular,
\begin{equation}\label{eq:hill}
R_{\mathrm{Hill}}=a_J\left(\frac{M_J}{3M_\odot}\right)^{1/3}.
\end{equation}
Con las constantes declaradas,
\begin{equation}
\SOI=48\,219\,247.59\ \mathrm{km},\qquad
R_{\mathrm{Hill}}=53\,152\,808.54\ \mathrm{km}
\quad (R_{\mathrm{Hill}}/\SOI\approx1.1023).
\end{equation}
Ninguno de los radios representa una superficie física abrupta donde desaparezca exactamente la influencia solar. Por ello se realiza un estudio de sensibilidad para distintos radios de empalme.

\section{Formulación del Modelo 1: cónicas empalmadas}
\subsection{Estado de entrada y cambio de marco}
La distancia real de la sonda a Júpiter se define por
\begin{equation}
d_J(t)=\bigl\lVert\rV^{\rm Horizons}(t)-\rJ^{\rm Horizons}(t)\bigr\rVert.
\end{equation}
El instante de entrada $t_{\rm in}$ se obtiene encontrando el primer cruce descendente $d_J(t)=\SOI$. Se inicializan las variables jovicéntricas en el cruce:
\begin{equation}\label{eq:entry}
\vect\rho(t_{\rm in})=\rV(t_{\rm in})-\rJ(t_{\rm in}),
\quad \vect w(t_{\rm in})=\vV(t_{\rm in})-\vJ(t_{\rm in}).
\end{equation}
Para el segmento interior se integra \emph{únicamente}
\begin{equation}\label{eq:model1}
\dot{\vect\rho}=\vect w,
\qquad
\dot{\vect w}=-\frac{\muJ}{\rho^3}\vect\rho.
\end{equation}
El evento terminal de salida corresponde al primer cruce ascendente
\begin{equation}
g_{\rm out}(t)=\lVert\vect\rho(t)\rVert-\SOI=0,
\quad \frac{\dif g_{\rm out}}{\dif t}>0.
\end{equation}
En el instante calculado $t_{\rm out}^{(1)}$ se transforman nuevamente las condiciones:
\begin{equation}\label{eq:exit}
\rV(t_{\rm out}^{(1)})=\rJ(t_{\rm out}^{(1)})+\vect\rho(t_{\rm out}^{(1)}),
\quad
\vV(t_{\rm out}^{(1)})=\vJ(t_{\rm out}^{(1)})+\vect w(t_{\rm out}^{(1)}).
\end{equation}
Se integran segmentos solares independientes antes y después del encuentro:
\begin{equation}
\ddot{\rV}=-\frac{\muS}{r^3}\rV.
\end{equation}
La trayectoria queda definida por tres soluciones empalmadas en $t_{\rm in}$ y $t_{\rm out}^{(1)}$. El estado de entrada es \emph{idéntico} para ambos modelos, mientras que sus fechas de salida pueden diferir y deben conservarse para interpretar el método.

\subsection{Conservaciones de la región jovicéntrica}
En el intervalo interior del Modelo 1, de la ecuación~\eqref{eq:model1} se deduce directamente
\begin{equation}\label{eq:inv1}
\boxed{\frac{\dif\varepsilon_J}{\dif t}=0,\qquad
\frac{\dif\vect h_J}{\dif t}=\vect0},
\quad
\varepsilon_J=\frac12 w^2-\frac{\muJ}{\rho},\quad
\vect h_J=\vect\rho\times\vect w.
\end{equation}
No se exige que la energía de Kepler alrededor del Sol sea constante \emph{durante} el segmento jovicéntrico, porque se está cambiando entre modelos y referencias. Tampoco se exige la continuidad de la aceleración en la frontera: sí son continuas posiciones y velocidades por construcción.

\section{Formulación del Modelo 2: Sol, Júpiter y sonda}
\subsection{Ecuación heliocéntrica restringida con término indirecto}
El Modelo 2 no integra la evolución gravitatoria de todos los cuerpos desde condiciones iniciales propias. La trayectoria real de Júpiter, $\rJ(t)$, es una función prescrita por Horizons. Se propaga únicamente una sonda de masa despreciable. Debido a que el origen está ligado al Sol, debe introducirse la aceleración de ese origen por Júpiter, llamada \emph{término indirecto}:
\begin{equation}\label{eq:model2}
\boxed{\ddot{\rV}=
-\muS\frac{\rV}{r^3}
-\muJ\left[
\frac{\rV-\rJ(t)}{|\rV-\rJ(t)|^3}
+\frac{\rJ(t)}{|\rJ(t)|^3}
\right].}
\end{equation}
El primer término corresponde a la atracción solar, el segundo a la atracción \emph{directa} de Júpiter, y el tercero a la corrección indirecta por la aceleración del Sol. La sonda recibe ambas contribuciones en todo el intervalo; aquí \textbf{no existe una frontera de cambio de ecuación}.

Si Júpiter fuese un primario dinámico del problema ideal de dos cuerpos Sol--Júpiter, la segunda ecuación sería
\begin{equation}
\ddot{\rJ}=-(\muS+\muJ)\frac{\rJ}{|\rJ|^3}.
\end{equation}
Pero en este proyecto $\rJ(t)$ se extrae de un sistema solar real y puede contener perturbaciones omitidas del modelo. La formulación es, por tanto, una \emph{aproximación restringida forzada por efemérides}, no una simulación autónoma completa de tres cuerpos.

\subsection{Fuerza diferencial, marea solar y derivadas de invariantes}
De $\vect\rho=\rV-\rJ$ se sigue
\begin{align}\label{eq:rel_acc}
\ddot{\vect\rho}
&=-\muJ\frac{\vect\rho}{\rho^3}+\vect a_{\rm marea}+\vect\delta_a,\\
\vect a_{\rm marea}
&= -\muS\left(\frac{\rV}{r^3}-\frac{\rJ}{|\rJ|^3}\right),\\
\vect\delta_a
&=-\muS\frac{\rJ}{|\rJ|^3}
-\muJ\frac{\rJ}{|\rJ|^3}
-\ddot{\rJ}_{\rm Horizons}.
\end{align}
La corrección $\vect\delta_a$ cuantifica la diferencia entre la aceleración joviana coherente con el modelo ideal Sol--Júpiter y la aceleración de las efemérides prescritas. En una órbita planetaria ideal compatible, $\vect\delta_a=\vect0$.

Para el Modelo 2, las variaciones de las cantidades jovicéntricas se obtienen sin aproximaciones adicionales a partir de la ecuación relativa:
\begin{align}\label{eq:derivE}
\boxed{\frac{\dif\varepsilon_J}{\dif t}=\vect w\cdot
(\vect a_{\rm marea}+\vect\delta_a)},\\
\boxed{\frac{\dif\vect h_J}{\dif t}=\vect\rho\times
(\vect a_{\rm marea}+\vect\delta_a)}.
\end{align}
El cociente adimensional que se traza es el diagnóstico de marea \emph{solar} respecto a la gravedad joviana directa,
\begin{equation}\label{eq:tidal}
\eta(t)=\frac{|\vect a_{\rm marea}|}{\muJ/\rho^2}.
\end{equation}
Este cociente no incluye $\vect\delta_a$ y no es un criterio universal de validez por sí solo; además, las pequeñas perturbaciones integradas pueden producir grandes diferencias a largo plazo.

Por su parte, la energía específica de Kepler heliocéntrica y el momento angular heliocéntrico satisfacen
\begin{align}
\varepsilon_\odot&=\frac12v^2-\frac{\muS}{r},
&\frac{\dif\varepsilon_\odot}{\dif t}
&=\vV\cdot\vect a_{J,\rm total},\\
\vect h_\odot&=\rV\times\vV,
&\frac{\dif\vect h_\odot}{\dif t}
&=\rV\times\vect a_{J,\rm total},
\end{align}
con
\begin{equation}
\vect a_{J,\rm total}=-\muJ\left(
\frac{\rV-\rJ}{|\rV-\rJ|^3}+\frac{\rJ}{|\rJ|^3}\right).
\end{equation}
Por tanto $\varepsilon_\odot$ y $\vect h_\odot$ tampoco son invariantes exactos del estado de la sonda bajo la fuerza combinada. No debe confundirse esta afirmación con la conservación total que tendría un \emph{sistema físico autónomo} de masas finitas.

\subsection{Constante de Jacobi y alcance de la comprobación}
En el \emph{problema restringido circular} de tres cuerpos, usando un sistema sinódico uniformemente rotante, existen un potencial efectivo $U_{\rm ef}$ y una constante
\begin{equation}\label{eq:jacobi}
C_J=2U_{\rm ef}(\vect r_{\rm rot})-\lVert\vect v_{\rm rot}\rVert^2.
\end{equation}
Para un sistema baricéntrico circular, con $\vect r_B,\vect v_B$ y velocidad angular constante $n\widehat{\vect n}$, esto se puede escribir en las variables inerciales baricéntricas como
\begin{equation}\label{eq:cstar}
C_J=2\left(\frac{\muS}{r_\odot}+\frac{\muJ}{r_J}\right)
-\lVert\vect v_B\rVert^2+2n\,\widehat{\vect n}\cdot(\vect r_B\times\vect v_B),
\end{equation}
con las distancias a los dos primarios tomadas desde sus posiciones baricéntricas. En el código $\vect r_B=\vect r-\chi\rJ$, $\vect v_B=\vect v-\chi\vJ$ y $\chi=\muJ/(\muS+\muJ)$, usando un eje y velocidad angular de referencia construidos localmente. Como Horizons no representa exactamente una órbita circular uniforme de Sol--Júpiter, el valor calculado se llama $C^*(t)$, \emph{cuasi-Jacobi}, y se usa exclusivamente como diagnóstico. Una prueba separada con órbita circular consistente permite comprobar que la fórmula tiende a conservarse en el caso ideal.

\section{Procedimiento computacional completo}
\subsection{Lectura y sincronización de JPL Horizons}
Los únicos archivos necesarios son \texttt{horizons\_jupiter.csv} y \texttt{horizons\_voyager-2.csv}. La rutina \texttt{horizons\_csv} acepta tanto el formato tabular limpio como la exportación con marcadores \texttt{\$\$SOE} y \texttt{\$\$EOE}. Comprueba el encabezado cuando está disponible: \emph{centro Sol}, \emph{unidades KM-S} y \emph{eclíptica J2000}. Se rechazan fechas inconsistentes.

Para interpolar \emph{posición y velocidad de forma compatible}, se utiliza un spline cúbico de Hermite, no dos interpoladores independientes. Para cada componente de posición entre dos muestras $t_k,t_{k+1}$, con $s=(t-t_k)/\Delta t$, se interpola
\begin{multline}\label{eq:hermite}
r(t)=(2s^3-3s^2+1)r_k+ (s^3-2s^2+s)\Delta t\,v_k\\
+(-2s^3+3s^2)r_{k+1}+(s^3-s^2)\Delta t\,v_{k+1},
\end{multline}
con $v(t)=\dif r/\dif t$. Este método preserva los valores y derivadas tabulados pero no inventa precisión física superior a la cadencia original de Horizons.

\subsection{Eventos geométricos: entrada, periapsis, salida y radio objetivo}
Primero se detectan índices donde $d_J-\SOI$ cambia de positivo a negativo para la entrada y viceversa para la salida. Cada cruce se refina mediante \texttt{brentq}; la fecha de máxima aproximación se calcula minimizando $d_J(t)$ con \texttt{minimize\_scalar} en el intervalo de la SOI. Para la proyección se busca el primer cruce ascendente de
\begin{equation}
g_{\rm obj}(t)=\lVert\rV(t)\rVert-R_{\rm obj},\quad
R_{\rm obj}=9.58\times149\,597\,870.7\ \mathrm{km}.
\end{equation}
Este evento no comprueba fase angular ni posición de Saturno, pues ese planeta no forma parte del modelo.

\subsection{Integración ODE y tolerancias}
En ambos modelos se escribe un sistema de primer orden de seis variables $\vect y=(x,y,z,v_x,v_y,v_z)$ y se usa \texttt{scipy.integrate.solve\_ivp} con \texttt{method='DOP853'} (Runge--Kutta explícito de orden alto), \texttt{dense\_output=True}, $\mathrm{rtol}=3\times10^{-10}$ y tolerancias absolutas separadas para posición $10^{-3}\,\mathrm{km}$ y velocidad $2\times10^{-11}\,\mathrm{km/s}$. El paso máximo nominal es de $0.75$ días; el integrador acorta automáticamente los pasos cerca del periapsis.

\begin{lstlisting}[caption={Ecuaciones de propagación: código abreviado, idéntico a las fuerzas usadas.}]
def solar_rhs(phys):
    def f(t, y):
        r, v = y[:3], y[3:]
        return np.r_[v, -phys.mu_sun*r / np.linalg.norm(r)**3]
    return f

def jovian_rhs(phys):
    def f(t, y):
        rho, w = y[:3], y[3:]
        return np.r_[w, -phys.mu_jupiter*rho / np.linalg.norm(rho)**3]
    return f

def planet_rhs(phys, eph):
    def f(t, y):
        r, v = y[:3], y[3:]
        rj = eph["jupiter"].state(t).r
        rho = r-rj
        a = -phys.mu_sun*r/np.linalg.norm(r)**3
        a -= phys.mu_jupiter*(rho/np.linalg.norm(rho)**3
                               + rj/np.linalg.norm(rj)**3)
        return np.r_[v, a]
    return f
\end{lstlisting}
\textbf{Nota:} el listado muestra las ecuaciones físicas de manera legible; las funciones ejecutables, la gestión de eventos y el programa \emph{completo} están incorporados en el cuaderno y disponibles en \texttt{asistencia\_gravitacional\_sol\_jupiter.py}.

\subsection{Esquema de flujo y relación con funciones Python}
\begin{enumerate}
\item \texttt{synchronize\_data}: leer únicamente Júpiter y Voyager y verificar mismo intervalo y malla temporal de Horizons.
\item \texttt{encounter\_real}: localizar $t_{\rm in}$, $t_{\rm peri}$ y $t_{\rm out}$ a partir de $d_J(t)$ y $\SOI$.
\item \texttt{conics\_orbit}: anclar el mismo estado de entrada y construir los tres arcos del Modelo 1: solar anterior, jovicéntrico, solar posterior.
\item \texttt{forced\_orbit}: con la misma entrada, integrar la ecuación~\eqref{eq:model2} hacia atrás y hacia adelante sin empalmes físicos.
\item \texttt{hyperbola\_geometry}: calcular $\varepsilon_J,h_J,v_\infty,b,e,D,\gamma,r_p$ a partir del estado jovicéntrico de entrada.
\item \texttt{berg\_parameters}: trasladar asintotas relativas a base tangencial--radial, calcular $\alpha$, $\beta$ y las ecuaciones (32), (37) y la comparación 3D.
\item \texttt{two\_body\_invariants}, \texttt{tidal\_ratio}, \texttt{jacobi\_proxy}: generar diagnósticos de conservación y perturbación.
\item \texttt{sensitivity\_study}: recalcular la entrada para distintos radios SOI y repetir ambas propagaciones con la \emph{misma} condición inicial en cada experimento.
\item \texttt{first\_radius\_time}: encontrar el primer cruce de $9.58\,\mathrm{UA}$ en cada modelo.
\item \texttt{make\_plots}: exportar once figuras de Matplotlib; \texttt{plotly\_animation\_2d} exporta el visor 2D; \texttt{generar\_panel3d} exporta las vistas interactivas jovicéntrica y heliocéntrica por Canvas 2D, sin WebGL.
\item \texttt{main}: escribir archivos CSV, resumen JSON y resultados, y mostrar verificaciones. El cuaderno Colab agrupa los productos en un ZIP descargable \emph{desde Colab}.
\end{enumerate}

\subsection{Definición de error y comparación en el mismo instante}
Para dos estados calculados en una misma época $t$ se utiliza
\begin{equation}
\delta_r(t)=\lVert\rV_1(t)-\rV_2(t)\rVert,
\qquad \delta_v(t)=\lVert\vV_1(t)-\vV_2(t)\rVert.
\end{equation}
Frente a Horizons se define de modo análogo $\delta_r^{\rm real}$ y $\delta_v^{\rm real}$. Si el Modelo 1 sale de la SOI antes que el Modelo 2, sus estados se evalúan \emph{ambos a la fecha de salida del Modelo 1}; no se deben restar dos posiciones tomadas en tiempos diferentes. El error de proyección se evalúa también en la misma \emph{fecha real de cruce} del radio $9.58$ UA. Las diferencias M1--M2 son el efecto de cambiar las ecuaciones aproximadas; las diferencias frente a Horizons también reflejan fuerzas que los modelos simplificados omiten.

\subsection{Sensibilidad al radio de empalme}
Se repite el procedimiento para
\begin{equation}
R_{\rm empalme}=f\,\SOI,
\quad f\in\left\{0.50,0.65,0.80,1.00,\frac{R_{\rm Hill}}{\SOI}\right\}.
\end{equation}
En cada caso se determina una nueva fecha de entrada sobre Horizons y se inicializan ambos modelos con \emph{ese mismo estado real}. La comparación no equivale a controlar todos los errores físicos, pero permite aislar la sensibilidad del esquema de cónicas empalmadas ante una decisión convencional del analista.

\section{Resultados obtenidos con los archivos del proyecto}
\subsection{Reconstrucción del encuentro real}
Los valores obtenidos con los CSV de Horizons y la interpolación Hermite son:
\begin{center}\begin{tabular}{ll}\toprule
Magnitud & Resultado\\\midrule
Entrada SOI Laplace & 6 de mayo de 1979, 11:34:09 TDB\\
Periapsis interpolado & 9 de julio de 1979, 22:26:54 TDB\\
Distancia mínima interpolada & $720\,894$ km\\
Salida SOI real & 12 de septiembre de 1979, 20:40:14 TDB\\
SOI Laplace / Hill & $48.219$ / $53.153$ millones de km\\
Cruce real de $9.58$ UA & 23 de agosto de 1981, 07:21:45 TDB\\
\bottomrule\end{tabular}\end{center}
El valor de periapsis depende de la interpolación y de la cadencia de los registros. No se presenta como una reconstrucción de navegación de alta precisión de la misión.

\subsection{Elementos heliocéntricos: antes y después}
Los elementos de la sonda reconstruidos a partir de Horizons en las fronteras reales de Laplace son:
\begin{center}\begin{tabular}{lrr}\toprule
Magnitud & Antes de la SOI & Después de la SOI\\\midrule
Energía específica $\varepsilon_\odot$ ($\mathrm{km^2/s^2}$) & $-121.446$ & $30.091$\\
Semieje mayor osculatriz $a$ (UA) & $3.652$ & $-14.741$\\
Excentricidad $e$ & $0.7371$ & $1.3406$\\
Afelio (UA) & $6.345$ & no existe\\
Inclinación eclíptica $i$ (grados) & $5.1205$ & $2.5799$\\\bottomrule\end{tabular}\end{center}
En la entrada el afelio osculatriz solar es inferior a $9.58\,\mathrm{UA}$; al salir, la cónica solar osculatriz es abierta ($\varepsilon_\odot>0$). Esto muestra la posibilidad de viajar a radios mayores sin impulso propulsivo adicional, siempre que se tenga en cuenta el movimiento de Júpiter.

\subsection{Parámetros analíticos de la hipérbola}
En el estado de entrada de Laplace:
\begin{center}\begin{tabular}{ll}\toprule
Parámetro & Valor\\\midrule
$\varepsilon_J$ & $29.43869\ \mathrm{km^2/s^2}$\\
$h_J$ & $1.269885\times10^7\ \mathrm{km^2/s}$\\
$v_\infty$ & $7.67316\ \mathrm{km/s}$\\
$b$ & $1\,654\,970\ \mathrm{km}$\\
$D$ & $1.300145$\\
Excentricidad $e_J$ & $1.26158$\\
Giro asintótico $\gamma$ & $104.869^\circ$\\
Periapsis de la cónica osculatriz & $562\,842\ \mathrm{km}$\\
$\alpha$ y $\beta$ proyectados según Berg & $53.337^\circ$ y $51.499^\circ$\\
Ganancia Berg (32) y (37) & $10.0076\ \mathrm{km/s}$\\
Ganancia asintótica vectorial 3D & $9.7525\ \mathrm{km/s}$\\\bottomrule\end{tabular}\end{center}
El periapsis de la hipérbola osculatriz pura difiere del periapsis de las efemérides reales: ello ilustra el efecto de fuerzas no incluidas en la aproximación. El ángulo de $104.869^\circ$ corresponde a la \emph{cónica jovicéntrica osculatriz en la SOI}, no al ángulo histórico nominal de unos $80^\circ$ mencionado por Berg como aproximación geométrica; no deben identificarse sin revisar las definiciones y condiciones iniciales.

\subsection{Ejemplo completo de sustitución numérica}
Para mostrar de dónde surgen los valores y permitir reproducirlos manualmente, tomamos el estado relativo \emph{interpolado} en $t_{\rm in}$ y las constantes de Júpiter. El código calcula primero
\begin{align*}
\varepsilon_J &=\frac12\lVert\vect w_0\rVert^2-\frac{\muJ}{\lVert\vect\rho_0\rVert}
 \approx29.43869\ \mathrm{km^2/s^2},\\
h_J&=\lVert\vect\rho_0\times\vect w_0\rVert
 \approx1.269885\times10^7\ \mathrm{km^2/s}.
\end{align*}
Una vez obtenidos estos \emph{dos invariantes osculatrices} se suceden las sustituciones
\begin{align*}
v_\infty&=\sqrt{2(29.43869)}\approx7.67316\ \mathrm{km/s},\\
b&=\frac{1.269885\times10^7}{7.67316}
 \approx1.65497\times10^6\ \mathrm{km},\\
D&=\frac{1.26686534\times10^8}{(1.65497\times10^6)(7.67316)^2}
 \approx1.300145,\\
\gamma&=2\arctan(1.300145)\approx104.869^\circ,\\
e_J&=\sqrt{1+\frac{2(29.43869)(1.269885\times10^7)^2}
 {(1.26686534\times10^8)^2}}\approx1.26158,\\
r_{p,\mathrm{osc}}&=\frac{(1.269885\times10^7)^2}{
(1.26686534\times10^8)(1+1.26158)}\approx5.62842\times10^5\ \mathrm{km}.
\end{align*}
El mismo procedimiento aplicado a cada estado heliocéntrico de Horizons da
\begin{align*}
 a_{\rm antes}&=-\frac{\muS}{2\varepsilon_{\odot,\rm antes}}
 =-\frac{1.32712440018\times10^{11}}{2(-121.446)}
 \approx 5.464\times10^8\ \mathrm{km}\approx3.652\ \mathrm{UA},\\
 r_{a,\rm antes}&=a_{\rm antes}(1+e_{\rm antes})
 \approx3.652(1+0.7371)=6.345\ \mathrm{UA},\\
 a_{\rm despues}&=-\frac{1.32712440018\times10^{11}}{2(30.0909)}
 \approx-14.741\ \mathrm{UA},\\
 \Delta\varepsilon_\odot&=\varepsilon_{\odot,\rm despues}-
 \varepsilon_{\odot,\rm antes}
 =30.0909-(-121.4460)\approx151.537\ \mathrm{km^2/s^2}.
\end{align*}
Estos cálculos son \emph{instantáneos u osculatrices}; las posiciones, velocidades, cruces y errores a fechas posteriores requieren la propagación numérica. Las cifras redondeadas aquí no deben reemplazar los valores de doble precisión usados por el código.

\subsection{Cuantificación de los dos modelos}
Los dos modelos empiezan en la misma época y estado de entrada. Para cada uno se mide la energía entre esa entrada y su \emph{propia} salida, y el error respecto a Horizons se calcula en esa misma fecha de salida:
\begin{center}\begin{tabular}{lrr}\toprule
Resultado & Modelo 1 & Modelo 2\\\midrule
$\Delta\varepsilon_\odot$ ($\mathrm{km^2/s^2}$) & $160.4714$ & $152.2056$\\
Error de posición al salir (km) & $6\,845\,903$ & $469\,583$\\
Error de velocidad al salir ($\mathrm{km/s}$) & $1.1256$ & $0.08715$\\
Cruce radial de $9.58$ UA & 10--08--1981 & 19--08--1981\\\bottomrule\end{tabular}\end{center}
Para comparación, Horizons da $\Delta\varepsilon_\odot\approx151.537\,\mathrm{km^2/s^2}$ entre \emph{sus propias} fronteras. La coincidencia de energía no es una prueba suficiente de exactitud orbital: se deben verificar también vector posición y vector velocidad.

\subsection{¿Qué se conserva realmente?}
En el Modelo 1, calculando la diferencia extrema de energía jovicéntrica dentro del tramo de dos cuerpos y las variaciones relativas del momento angular vectorial,
\begin{equation}
\operatorname{rango}(\varepsilon_J)\approx6.57\times10^{-8}\ \mathrm{km^2/s^2},
\quad \max_t\frac{|\vect h_J(t)-\vect h_J(0)|}{|\vect h_J(0)|}
\approx1.15\times10^{-10}.
\end{equation}
Son residuos de integración numérica. En el Modelo 2 la variación total de $\varepsilon_J$ en la ventana común llega a $0.275\,\mathrm{km^2/s^2}$ y el cambio relativo vectorial máximo de $\vect h_J$ es cercano a $15.2\%$. Estas variaciones son físicas dentro del modelo forzado y reflejan la presencia de aceleraciones no centrales jovicéntricas. No indican que se pierda energía en el universo.

El cociente $\eta$ de la ecuación~\eqref{eq:tidal} varía en la ventana analizada entre $7.76\times10^{-7}$ y $0.4267$. La perturbación relativa es muy pequeña cerca del planeta, pero considerable en la periferia de la SOI. Los umbrales ilustrativos $\delta_r\geq10^3\,\mathrm{km}$, $\delta_r\geq10^5\,\mathrm{km}$ y $\delta_v\geq0.1\,\mathrm{km/s}$ aparecen aproximadamente el 9 de mayo, 12 de junio y 4 de julio de 1979, respectivamente. \emph{Son convenciones diagnósticas, no un instante universal de ruptura de equivalencia.}

\subsection{Resultados del estudio de sensibilidad}
\begin{center}\begin{tabular}{rrr}\toprule
Radio $f\,\SOI$ & Error M1--M2 a salida (millones km) & Diferencia proyectada a 9.58 UA (millones km)\\\midrule
$0.50$ & $0.374$ & $39.358$\\
$0.65$ & $1.182$ & $32.296$\\
$0.80$ & $2.771$ & $40.280$\\
$1.00$ & $7.190$ & $76.182$\\
Hill ($1.102$) & $10.598$ & $102.087$\\\bottomrule\end{tabular}\end{center}
El error proyectado mínimo de este conjunto discreto se obtiene para $f=0.65$; no hay garantía de que sea un óptimo físico universal, ni necesariamente el mejor valor para otro objetivo, época o sistema.

\subsection{Figuras del cálculo}\label{sec:figuras}
\fign{01_trayectoria_jovicentrica.png}{.96}{Reconstrucción de la trayectoria cerca de Júpiter para Horizons, cónicas empalmadas y propagación simultánea.}
\fign{04_invariantes_energia_momento.png}{.96}{Energías específicas y momentos angulares en los dos marcos de referencia.}
\fign{06_marea_y_error.png}{.90}{Relación entre marea solar, aceleración joviana y divergencia de los modelos.}
\fign{08_trayectoria_radio_objetivo.png}{.85}{Proyección heliocéntrica; $9.58$ UA es un radio de referencia, no una órbita planetaria calculada.}
\fign{10_sensibilidad_soi.png}{.93}{Errores frente a los distintos radios de empalme, comparados en tiempos comunes.}

\section{Descripción y organización del código}
El proyecto se puede ejecutar por completo en Google Colab sin modificar archivos de sistema ni recurrir a GPU. El cuaderno incluye el motor científico en una celda propia, el análisis, la generación de once figuras PNG, las tablas CSV, la animación Plotly 2D y dos HTML independientes con Canvas 2D. La cámara interactiva se centra de forma predeterminada en Júpiter; se puede seleccionar seguimiento de sonda o Sol, y trasladar la vista con \emph{Mayús + arrastre} o botón derecho. No se utiliza \texttt{go.Scatter3d}, que requeriría WebGL.

\textbf{Arquitectura:} \texttt{Physics} agrupa parámetros físicos, \texttt{Settings} tolerancias y opciones, \texttt{Ephemeris} encapsula datos e interpolación, \texttt{State} describe cada estado físico, \texttt{Encounter} agrupa eventos geométricos, y \texttt{Orbit} evalúa tramos integrados. Funciones separadas implementan fuerzas, integraciones, métricas, diagnósticos y figuras. Los resultados se exportan en \texttt{comparacion\_modelos.csv}, \texttt{elementos\_osculatrices.csv}, \texttt{sensibilidad\_radio\_soi.csv}, \texttt{diagnosticos\_temporales.csv}, \texttt{resumen\_fisico.json} y \texttt{telemetria\_dashboard\_*.csv}.

\textbf{Procedimiento reproducible en Colab:}
\begin{enumerate}
\item Abrir \texttt{Voyager2\_RetoIII3\_SOLO\_SOL\_JUPITER.ipynb}.
\item Ejecutar la celda de dependencias y cargar únicamente los CSV de Júpiter y Voyager 2.
\item Ejecutar la celda del motor físico, y después la celda de \texttt{main()}.
\item Ejecutar las dos celdas de visualización HTML, una jovicéntrica y otra heliocéntrica.
\item Ejecutar la última celda para generar un ZIP y \emph{descargarlo desde Colab}. El archivo LaTeX y su PDF de lectura pueden conservarse en la misma carpeta del proyecto y se incluyen como documentación externa.
\end{enumerate}

\subsection{Limitaciones y comprobaciones de consistencia}
\begin{itemize}
\item Las efemérides históricas de Horizons incluyen fuerzas y características dinámicas omitidas en los dos modelos; el error contra Horizons \emph{no mide exclusivamente} el error de cónicas empalmadas. Para comparar aproximaciones en igualdad de condiciones, también se computa M1--M2.
\item El movimiento de Júpiter se toma de Horizons, mientras el modelo de la sonda solo incorpora gravedad solar y joviana. Esto es una aproximación forzada, no una integración simultánea autoconsistente de todas las masas.
\item Se trata el Sol como centro heliocéntrico con corrección indirecta; esta corrección evita mezclar sin aviso un origen acelerado con ecuaciones inerciales.
\item El parámetro de SOI y la malla temporal son decisiones de modelado. Las fechas subdiarias calculadas por interpolación no deben confundirse con mediciones originales diarias.
\item La conservación de Jacobi requiere una órbita circular coherente y un sistema rotante uniforme. Con efemérides reales solo se grafica $C^*(t)$ aproximada; una autoprueba con órbita circular ideal recupera deriva relativa del orden de $10^{-10}$.
\item Cruzar el radio $9.58\,\mathrm{UA}$ no prueba una cita orbital con Saturno. No se usan ni su posición ni su gravedad en el proyecto final.
\end{itemize}

\section{Conclusiones}
\begin{enumerate}[label=\textbf{\arabic*.}]
\item La dispersión por fuerza central proporciona una comprensión analítica del encuentro: la dirección de la velocidad relativa cambia con un giro que depende de $D=\muJ/(bv_\infty^2)$; su energía relativa y el momento angular vectorial se conservan en la descripción jovicéntrica de dos cuerpos.
\item La asistencia gravitacional no genera energía de la nada. El cambio de rapidez heliocéntrica proviene de la transformación de velocidades y del intercambio de energía orbital con Júpiter: $\Delta K_\odot\simeq\vJ\cdot(\vect w_+-\vect w_-)$.
\item El Modelo 1 reproduce el mecanismo esencial con un costo numérico bajo y conserva $\varepsilon_J$ y $\vect h_J$ en el segmento de la SOI. Sin embargo, su hipótesis elimina la marea solar durante un intervalo largo, lo que se acumula como error de trayectoria.
\item El Modelo 2, limitado a Sol--Júpiter--sonda, reproduce mejor las efemérides reales de salida: aproximadamente $0.470$ millones de km de error posicional frente a $6.846$ millones de km para el Modelo 1. La discrepancia no se explica por el integrador, sino principalmente por el modelo físico empleado.
\item No existe un instante único y universal cuando las aproximaciones ``dejan de ser equivalentes''. La respuesta depende de la tolerancia de error aceptable, del radio de empalme, de la duración de la predicción y de la magnitud comparada.
\item La energía heliocéntrica osculatriz se vuelve positiva después del paso joviano y la trayectoria alcanza el radio geométrico $9.58\,\mathrm{UA}$. Este resultado satisface la condición radial pedida sin introducir interacciones gravitatorias adicionales.
\end{enumerate}

\section{Referencias}
\begin{thebibliography}{9}
\bibitem{berg2023} P. Berg, ``The fundamental concepts of the gravity-assist manoeuvre'', \emph{European Journal of Physics}, vol.~44, art.~025002, 2023. DOI: \href{https://doi.org/10.1088/1361-6404/acaad8}{10.1088/1361-6404/acaad8}. Se utilizó la traducción al español suministrada para el proyecto, especialmente secciones 2--5 y ecuaciones (26)--(27), (32), (37), (39)--(42).
\bibitem{horizons} NASA/JPL, \emph{Horizons On-Line Ephemeris System}. Archivos CSV históricos de Júpiter y Voyager 2 aportados para este proyecto; marco de referencia y unidades según sus encabezados.
\bibitem{scipy} SciPy, documentación de \texttt{solve\_ivp} (método DOP853), \texttt{CubicHermiteSpline} y \texttt{brentq}; implementaciones utilizadas por el programa.
\end{thebibliography}
\end{document}
'''
RUTA_TEX.write_text(TEX_FUENTE, encoding='utf-8')
FIGURAS = SALIDA / 'figuras'
FIGURAS.mkdir(exist_ok=True)
for png in SALIDA.glob('*.png'):
    shutil.copy2(png, FIGURAS / png.name)
print('LaTeX guardado:', RUTA_TEX)
print('Figuras disponibles para compilar:', len(list(FIGURAS.glob('*.png'))))

LaTeX guardado: /content/resultados_voyager2_sol_jupiter/Informe_Reto_III3_Sol_Jupiter_Voyager2.tex
Figuras disponibles para compilar: 11


## 8. Descargar todos los productos desde Google Colab

Esta celda crea un ZIP directamente en tu sesión de Colab: HTML jovicéntrico y heliocéntrico, animación Plotly 2D, 11 figuras PNG, CSV de las dos vistas y del estudio, resumen JSON, **documento LaTeX** y sus imágenes. El cuaderno `.ipynb` ya incluye el motor científico, así que no requiere un archivo Python separado.

In [10]:
# 8. Generar índice y descargar paquete ZIP CON todos los resultados
import zipfile
from pathlib import Path
from html import escape
from google.colab import files

SALIDA=Path('/content/resultados_voyager2_sol_jupiter')
assert (SALIDA/'13_sobrevuelo_jupiter_centrado.html').exists(), 'Ejecuta la celda HTML jovicéntrico.'
assert (SALIDA/'12_trayectoria_heliocentrica_sin_saturno.html').exists(), 'Ejecuta la celda HTML heliocéntrico.'
assert (SALIDA/'Informe_Reto_III3_Sol_Jupiter_Voyager2.tex').exists(), 'Ejecuta la celda LaTeX.'
htmls=sorted(SALIDA.glob('*.html'))
figs=sorted(SALIDA.glob('*.png'))
tablas=sorted(SALIDA.glob('*.csv'))
def enlaces(archivos):
    return '\n'.join(f'<li><a href="{escape(p.name,quote=True)}">{escape(p.name)}</a></li>' for p in archivos)
indice=SALIDA/'00_ABRIR_RESULTADOS.html'
indice.write_text(f'''<!DOCTYPE html><html lang="es"><head><meta charset="utf-8"><title>Voyager 2 - Reto III.3</title>
<style>body{{font:16px system-ui;background:#101929;color:#eaf2ff;max-width:950px;margin:3em auto;padding:1em}}a{{color:#8fdbff}}li{{padding:.3em}}</style></head><body>
<h1>Voyager 2 · Sol–Júpiter–sonda</h1><p>Dos modelos físicos. Sin interacción con Saturno. Radio de 9.58 UA como referencia.</p>
<h2>Simulaciones HTML</h2><ul>{enlaces(htmls)}</ul><h2>Figuras</h2><ul>{enlaces(figs)}</ul>
<h2>Tablas CSV</h2><ul>{enlaces(tablas)}</ul>
<p>Informe en LaTeX: Informe_Reto_III3_Sol_Jupiter_Voyager2.tex</p>
</body></html>''', encoding='utf-8')
(SALIDA/'LEEME_PRIMERO.txt').write_text(
    'RETO III.3 · SOL, JÚPITER Y VOYAGER 2\n'
    '1. Abre 00_ABRIR_RESULTADOS.html en un navegador.\n'
    '2. Sobrevuelo centrado en Júpiter: 13_sobrevuelo_jupiter_centrado.html\n'
    '3. Vista heliocéntrica: 12_trayectoria_heliocentrica_sin_saturno.html\n'
    '4. Para Overleaf: sube Informe_Reto_III3_Sol_Jupiter_Voyager2.tex y la carpeta figuras/\n'
    '5. El archivo de entrada de Saturno NO se usa. 9.58 UA es un radio geométrico.\n',
    encoding='utf-8')
zip_final=Path('/content/Voyager2_Sol_Jupiter_Resultados_Completos.zip')
with zipfile.ZipFile(zip_final,'w',compression=zipfile.ZIP_DEFLATED,compresslevel=6) as zf:
    for p in sorted(SALIDA.rglob('*')):
        if p.is_file(): zf.write(p,arcname='Voyager2_Sol_Jupiter/'+p.relative_to(SALIDA).as_posix())
print('Paquete creado:',zip_final, 'tamaño MB:',round(zip_final.stat().st_size/1e6,2))
print('Archivos:',len(zipfile.ZipFile(zip_final).namelist()))
files.download(str(zip_final))

Paquete creado: /content/Voyager2_Sol_Jupiter_Resultados_Completos.zip tamaño MB: 5.27
Archivos: 35


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>